# Load data & index semua file

In [1]:
# ============================================================
# STAGE B — LOAD DATA & INDEX SEMUA FILE (ONE CELL) [REVISI FULL: FIX list-column assignment]
# Output utama:
# - PATHS (dict) termasuk PATHS["dinov2_model_dir"]
# - df_index  : index semua image (train + supplemental + test)
# - df_train  : subset train
# - df_test   : subset test
# - artifacts/df_index.parquet (atau .csv kalau parquet gagal)
# ============================================================

import os, re
from pathlib import Path
import pandas as pd
import numpy as np

# ----------------------------
# PATHS (dataset + model)
# ----------------------------
ROOT = "/kaggle/input/recodai-luc-scientific-image-forgery-detection"
DINO_DIR = "/kaggle/input/dinov2/pytorch/small/1"

PATHS = {
    "root": ROOT,
    "train_images": f"{ROOT}/train_images",
    "train_masks": f"{ROOT}/train_masks",
    "supp_images": f"{ROOT}/supplemental_images",
    "supp_masks": f"{ROOT}/supplemental_masks",
    "test_images": f"{ROOT}/test_images",
    "sample_submission": f"{ROOT}/sample_submission.csv",
    "dinov2_model_dir": DINO_DIR,
    "work_root": "/kaggle/working/recodai_luc",
    "artifact_dir": "/kaggle/working/recodai_luc/artifacts",
}
os.makedirs(PATHS["artifact_dir"], exist_ok=True)

# quick sanity check model files (tidak load model)
dino_req = ["config.json", "preprocessor_config.json", "pytorch_model.bin"]
missing_dino = [f for f in dino_req if not Path(PATHS["dinov2_model_dir"], f).exists()]
print("===== DINOv2 PATH CHECK =====")
print("dinov2_model_dir:", PATHS["dinov2_model_dir"])
if missing_dino:
    print("WARNING: file DINOv2 kurang:", missing_dino)
else:
    print("OK: config/preprocessor/pytorch_model.bin ditemukan")

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
MASK_EXTS = {".npy"}

def list_files(root, exts):
    root = Path(root)
    if not root.exists():
        return []
    out = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts:
            out.append(str(p))
    return sorted(out)

def infer_is_forged_from_path(p):
    lp = p.lower().replace("\\", "/")
    if "/authentic/" in lp:
        return 0
    if "/forged/" in lp:
        return 1
    return None

def case_id_from_path(p):
    return Path(p).stem

# ----------------------------
# 1) LOAD SAMPLE SUBMISSION (untuk case_id test)
# ----------------------------
if not Path(PATHS["sample_submission"]).exists():
    raise FileNotFoundError(f"sample_submission.csv tidak ditemukan: {PATHS['sample_submission']}")

df_sub = pd.read_csv(PATHS["sample_submission"])
if "case_id" not in df_sub.columns:
    raise ValueError(f"Kolom 'case_id' tidak ada di sample_submission. Kolom ada: {list(df_sub.columns)}")
test_case_ids = df_sub["case_id"].astype(str).tolist()

# ----------------------------
# 2) INDEX IMAGE FILES (train + supplemental + test)
# ----------------------------
train_img_paths = list_files(PATHS["train_images"], IMG_EXTS)
supp_img_paths  = list_files(PATHS["supp_images"],  IMG_EXTS)
test_img_paths  = list_files(PATHS["test_images"],  IMG_EXTS)

train_all_paths = train_img_paths + supp_img_paths

rows = []
seen = {}  # case_id -> img_path (dedupe)

def add_image_rows(paths, split_name):
    for p in paths:
        cid = str(case_id_from_path(p))
        if cid in seen:
            continue
        seen[cid] = p
        rows.append({
            "case_id": cid,
            "split": split_name,  # train / test
            "img_path": p,
            "is_forged": infer_is_forged_from_path(p),
            "source": "supplemental" if "/supplemental_images/" in p.replace("\\", "/") else "base",
        })

add_image_rows(train_all_paths, "train")
add_image_rows(test_img_paths, "test")

df_index = pd.DataFrame(rows)
if df_index.empty:
    raise RuntimeError("df_index kosong. Cek path ROOT dan folder dataset.")

# ----------------------------
# 3) INDEX MASK FILES (train_masks + supplemental_masks)
#    1 gambar bisa multi-mask -> list
# ----------------------------
mask_paths = list_files(PATHS["train_masks"], MASK_EXTS) + list_files(PATHS["supp_masks"], MASK_EXTS)

# simpan (stem, path)
mask_stems = [(str(Path(p).stem), p) for p in mask_paths]

# case_id train (unik)
train_case_ids = df_index.loc[df_index["split"].eq("train"), "case_id"].astype(str).tolist()
cid2masks = {cid: [] for cid in train_case_ids}

# heuristic cepat: bucket berdasar token awal sebelum '_' atau '-'
prefix2paths = {}
for stem, p in mask_stems:
    tok = re.split(r"[_-]", stem, maxsplit=1)[0]
    prefix2paths.setdefault(tok, []).append(p)

# assign masks ke setiap case_id train
for cid in list(cid2masks.keys()):
    candidates = prefix2paths.get(cid, [])
    rx = re.compile(rf"^{re.escape(cid)}($|[_-])")
    if candidates:
        hits = [p for p in candidates if rx.match(Path(p).stem)]
    else:
        hits = [p for stem, p in mask_stems if rx.match(stem)]
    cid2masks[cid] = sorted(hits)

# ----------------------------
# FIX UTAMA: isi kolom list-of-lists dengan dtype object (tanpa error pandas)
# ----------------------------
# init kolom mask_paths sebagai object, setiap baris list baru
df_index["mask_paths"] = pd.Series([[] for _ in range(len(df_index))], dtype=object)

# mapping semua case_id -> list mask (train), kalau bukan train tetap []
mask_series = df_index["case_id"].map(lambda x: cid2masks.get(str(x), []))

train_mask = df_index["split"].eq("train").to_numpy()
# assign sebagai ndarray dtype=object supaya pandas tidak mencoba jadi array 2D
df_index.loc[train_mask, "mask_paths"] = np.array(mask_series[train_mask].tolist(), dtype=object)

# isi is_forged kalau belum kebaca dari folder (fallback: ada mask => forged)
need_fill = df_index["split"].eq("train") & df_index["is_forged"].isna()
if need_fill.any():
    df_index.loc[need_fill, "is_forged"] = df_index.loc[need_fill, "mask_paths"].map(
        lambda lst: 1 if isinstance(lst, list) and len(lst) > 0 else 0
    )

# ----------------------------
# 4) VALIDASI RINGAN
# ----------------------------
df_train = df_index[df_index["split"].eq("train")].reset_index(drop=True)
df_test  = df_index[df_index["split"].eq("test")].reset_index(drop=True)

test_map = {cid: p for cid, p in zip(df_test["case_id"].astype(str).tolist(), df_test["img_path"].astype(str).tolist())}
missing_test = [cid for cid in test_case_ids if cid not in test_map]

print("\n===== INDEX SUMMARY =====")
print(f"Train images (base):        {len(train_img_paths)}")
print(f"Train images (supplement):  {len(supp_img_paths)}")
print(f"Train images (total):       {len(df_train)}")
print(f"Test images (found):        {len(df_test)}")
print(f"Masks (total .npy):         {len(mask_paths)}")
print(f"Test case_id in sample:     {len(test_case_ids)}")
print(f"Missing test images vs sample_submission: {len(missing_test)}")
if len(missing_test) > 0:
    print("Contoh missing (maks 10):", missing_test[:10])

print("\nTrain forged/authentic (infer):")
if "is_forged" in df_train.columns:
    print(df_train["is_forged"].value_counts(dropna=False))
else:
    print("Kolom is_forged tidak ada.")

print("\nSanity mask_paths:")
print("mask_paths dtype:", df_index["mask_paths"].dtype)
print("contoh (train)  :", df_train.loc[0, "case_id"], "n_masks=", len(df_train.loc[0, "mask_paths"]) if isinstance(df_train.loc[0, "mask_paths"], list) else "NA")
if len(df_test) > 0:
    print("contoh (test)   :", df_test.loc[0, "case_id"], "mask_paths=", df_test.loc[0, "mask_paths"])

print("\nHead df_index:")
print(df_index.head(10))

# ----------------------------
# 5) SAVE ARTIFACT
# ----------------------------
out_parq = os.path.join(PATHS["artifact_dir"], "df_index.parquet")
out_csv  = os.path.join(PATHS["artifact_dir"], "df_index.csv")

try:
    df_index.to_parquet(out_parq, index=False)
    print(f"\nSaved: {out_parq}")
except Exception as e:
    df_index.to_csv(out_csv, index=False)
    print(f"\nParquet gagal ({type(e).__name__}: {e}) -> Saved CSV: {out_csv}")


===== DINOv2 PATH CHECK =====
dinov2_model_dir: /kaggle/input/dinov2/pytorch/small/1
OK: config/preprocessor/pytorch_model.bin ditemukan

===== INDEX SUMMARY =====
Train images (base):        5128
Train images (supplement):  48
Train images (total):       2795
Test images (found):        1
Masks (total .npy):         2799
Test case_id in sample:     1
Missing test images vs sample_submission: 0

Train forged/authentic (infer):
is_forged
0.0    2377
1.0     418
Name: count, dtype: int64

Sanity mask_paths:
mask_paths dtype: object
contoh (train)  : 10 n_masks= 1
contoh (test)   : 45 mask_paths= []

Head df_index:
  case_id  split                                           img_path  \
0      10  train  /kaggle/input/recodai-luc-scientific-image-for...   
1   10015  train  /kaggle/input/recodai-luc-scientific-image-for...   
2   10017  train  /kaggle/input/recodai-luc-scientific-image-for...   
3   10030  train  /kaggle/input/recodai-luc-scientific-image-for...   
4   10070  train  /kaggle

# Bangun ground-truth mask “union” + “instance list”

In [2]:
# ============================================================
# STAGE C — BUILD GROUND-TRUTH (GT) MASK: "UNION" + "INSTANCE LIST" (ONE CELL)
# Prasyarat:
# - PATHS sudah ada (dari Stage B)
# - df_index, df_train sudah ada (atau akan diload dari artifacts/df_index.parquet)
#
# Output:
# - gt_cache : cache in-memory {case_id: (gt_union, gt_instances)}
# - load_gt(case_id) -> (gt_union, gt_instances)
#   * gt_union:  (H,W) uint8 {0,1}
#   * gt_instances: list[(H,W) uint8 {0,1}] (bisa kosong)
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

# ----------------------------
# Ensure df_index tersedia
# ----------------------------
if "df_index" not in globals() or not isinstance(df_index, pd.DataFrame):
    # coba load dari artifact Stage B
    art = Path(PATHS["artifact_dir"]) / "df_index.parquet"
    art_csv = Path(PATHS["artifact_dir"]) / "df_index.csv"
    if art.exists():
        df_index = pd.read_parquet(art)
    elif art_csv.exists():
        df_index = pd.read_csv(art_csv)
    else:
        raise RuntimeError("df_index tidak ada. Jalankan Stage B dulu.")

df_train = df_index[df_index["split"].eq("train")].reset_index(drop=True)
df_test  = df_index[df_index["split"].eq("test")].reset_index(drop=True)

# mapping cepat
cid2img = dict(zip(df_train["case_id"].astype(str), df_train["img_path"].astype(str)))
cid2mps = dict(zip(df_train["case_id"].astype(str), df_train["mask_paths"]))

# ----------------------------
# Helpers
# ----------------------------
def _read_image_shape(img_path):
    im = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(f"Gagal baca image: {img_path}")
    if im.ndim == 2:
        return im.shape[0], im.shape[1]
    return im.shape[0], im.shape[1]

def _load_npy_mask(mask_path):
    m = np.load(mask_path)
    # handle shape: (H,W) or (H,W,1) or (1,H,W)
    m = np.array(m)
    if m.ndim == 3:
        # ambil channel pertama yang masuk akal
        if m.shape[-1] == 1:
            m = m[..., 0]
        elif m.shape[0] == 1:
            m = m[0, ...]
        else:
            # kalau multi-channel, ambil max across channel
            m = m.max(axis=-1)
    if m.ndim != 2:
        # fallback: squeeze
        m = np.squeeze(m)
        if m.ndim != 2:
            raise ValueError(f"Mask shape tidak didukung: {m.shape} | file={mask_path}")
    # binarize
    if m.dtype != np.uint8:
        m = (m > 0).astype(np.uint8)
    else:
        m = (m > 0).astype(np.uint8)
    return m

def _align_mask_to_shape(mask_u8, H, W):
    if mask_u8.shape[0] == H and mask_u8.shape[1] == W:
        return mask_u8
    # resize nearest agar biner tidak rusak
    return cv2.resize(mask_u8, (W, H), interpolation=cv2.INTER_NEAREST).astype(np.uint8)

# ----------------------------
# GT Loader with cache
# ----------------------------
gt_cache = {}

def load_gt(case_id, align_to_image=True, drop_empty_instances=True):
    """
    Returns:
      gt_union: (H,W) uint8 {0,1}
      gt_instances: list of (H,W) uint8 {0,1}
    """
    cid = str(case_id)
    if cid in gt_cache:
        return gt_cache[cid]

    if cid not in cid2img:
        raise KeyError(f"case_id {cid} tidak ada di df_train (GT hanya untuk train).")

    img_path = cid2img[cid]
    H, W = _read_image_shape(img_path) if align_to_image else (None, None)

    mpaths = cid2mps.get(cid, [])
    if not isinstance(mpaths, list):
        # kalau kebaca sebagai string di CSV
        # coba parse sederhana: "['a','b']"
        try:
            import ast
            mpaths = ast.literal_eval(mpaths)
        except Exception:
            mpaths = []

    instances = []
    for mp in mpaths:
        if not mp or not Path(mp).exists():
            continue
        m = _load_npy_mask(mp)
        if align_to_image:
            m = _align_mask_to_shape(m, H, W)
        if drop_empty_instances and m.sum() == 0:
            continue
        instances.append(m)

    # kalau tidak ada mask, gt_union = all-zero
    if align_to_image:
        gt_union = np.zeros((H, W), dtype=np.uint8)
    else:
        # jika tidak align, union shape ikut mask pertama (kalau ada)
        gt_union = np.zeros_like(instances[0], dtype=np.uint8) if instances else np.zeros((1, 1), dtype=np.uint8)

    if instances:
        # union OR
        u = instances[0].copy()
        for m in instances[1:]:
            u = np.maximum(u, m)
        gt_union = (u > 0).astype(np.uint8)

    gt_cache[cid] = (gt_union, instances)
    return gt_union, instances

# ----------------------------
# Quick sanity check (ringan)
# ----------------------------
print("===== STAGE C: GT READY =====")
print(f"df_train: {len(df_train)} | df_test: {len(df_test)}")

# ambil beberapa contoh
sample_forged = df_train[df_train["is_forged"].astype(int).eq(1)].head(3)["case_id"].astype(str).tolist() if "is_forged" in df_train.columns else []
sample_auth   = df_train[df_train["is_forged"].astype(int).eq(0)].head(3)["case_id"].astype(str).tolist() if "is_forged" in df_train.columns else []

samples = (sample_forged + sample_auth)[:6]
for cid in samples:
    u, inst = load_gt(cid)
    print(f"- case_id={cid} | instances={len(inst)} | union_area={int(u.sum())} | shape={u.shape}")


===== STAGE C: GT READY =====
df_train: 2795 | df_test: 1
- case_id=10307 | instances=1 | union_area=163200 | shape=(1200, 1600)
- case_id=10505 | instances=1 | union_area=17580 | shape=(102, 986)
- case_id=1070 | instances=1 | union_area=63162 | shape=(520, 696)
- case_id=10 | instances=1 | union_area=928 | shape=(512, 648)
- case_id=10015 | instances=1 | union_area=3747 | shape=(1200, 1600)
- case_id=10017 | instances=1 | union_area=1175 | shape=(256, 320)


# Buat validasi internal yang “aman”

In [3]:
# ============================================================
# STAGE D — BUAT VALIDASI INTERNAL "AMAN" (ONE CELL)
# Tujuan:
# - Membuat kolom fold pada df_train (K-Fold) yang mengurangi leakage
# - Default: StratifiedKFold by is_forged (kalau tersedia)
# - Tambahan "group-ish" heuristic: pHash image untuk mencegah near-duplicate
#
# Prasyarat:
# - PATHS, df_index, df_train sudah ada (Stage B)
# - load_gt tersedia (Stage C) (opsional untuk isi is_forged kalau kosong)
#
# Output:
# - df_index (updated) dengan kolom:
#   * fold (int) untuk split=train
#   * fold_key / group_id (optional)
# - df_train, df_test (updated)
# - artifacts/df_index_with_folds.parquet (atau .csv)
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

# sklearn split
from sklearn.model_selection import StratifiedKFold, KFold

# ----------------------------
# Ensure df_index tersedia
# ----------------------------
if "df_index" not in globals() or not isinstance(df_index, pd.DataFrame):
    art = Path(PATHS["artifact_dir"]) / "df_index.parquet"
    art_csv = Path(PATHS["artifact_dir"]) / "df_index.csv"
    if art.exists():
        df_index = pd.read_parquet(art)
    elif art_csv.exists():
        df_index = pd.read_csv(art_csv)
    else:
        raise RuntimeError("df_index tidak ada. Jalankan Stage B dulu.")

df_train = df_index[df_index["split"].eq("train")].reset_index(drop=True)
df_test  = df_index[df_index["split"].eq("test")].reset_index(drop=True)

# ----------------------------
# CONFIG VALIDASI
# ----------------------------
SEED = 2025
N_FOLDS = 5

# gunakan pHash untuk grouping near-duplicate (aman dan cepat)
USE_PHASH_GROUP = True
PHASH_SIZE = 8               # 8x8 -> 64-bit
PHASH_MAX_DIST = 4           # <=4 dianggap satu grup (agresif tapi membantu)
PHASH_MAX_SIDE = 512         # resize agar cepat di CPU

# ----------------------------
# Helper: pHash (DCT-based) -> uint64
# ----------------------------
def _read_gray_u8(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return None
    if img.ndim == 3:
        if img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if img.dtype != np.uint8:
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return img

def phash64(gray_u8, hash_size=8, highfreq_factor=4):
    # output uint64
    if gray_u8 is None:
        return 0
    img = gray_u8
    h, w = img.shape[:2]
    s = max(h, w)
    if s > PHASH_MAX_SIDE:
        sc = PHASH_MAX_SIDE / float(s)
        img = cv2.resize(img, (int(w*sc), int(h*sc)), interpolation=cv2.INTER_AREA)

    # resize to (hash_size*highfreq_factor)
    sz = hash_size * highfreq_factor
    img = cv2.resize(img, (sz, sz), interpolation=cv2.INTER_AREA).astype(np.float32)
    dct = cv2.dct(img)
    dct_low = dct[:hash_size, :hash_size]
    med = np.median(dct_low[1:, 1:])  # skip DC often
    bits = (dct_low > med).astype(np.uint8).flatten()

    v = 0
    for b in bits:
        v = (v << 1) | int(b)
    return np.uint64(v)

def hamming64(a, b):
    return int(bin(int(a ^ b)).count("1"))

# ----------------------------
# Pastikan is_forged ada untuk stratify (kalau belum ada, isi via mask_paths)
# ----------------------------
if "is_forged" not in df_train.columns:
    df_train["is_forged"] = np.nan

need = df_train["is_forged"].isna()
if need.any():
    # fallback: forged jika punya mask_paths
    def _has_mask(x):
        if isinstance(x, list):
            return 1 if len(x) > 0 else 0
        # kalau string dari CSV, coba parse
        try:
            import ast
            lst = ast.literal_eval(x)
            return 1 if isinstance(lst, list) and len(lst) > 0 else 0
        except Exception:
            return 0
    df_train.loc[need, "is_forged"] = df_train.loc[need, "mask_paths"].map(_has_mask)

df_train["is_forged"] = df_train["is_forged"].astype(int)

# ----------------------------
# Build group_id dengan pHash (optional)
# ----------------------------
if USE_PHASH_GROUP:
    print("Building pHash groups (near-duplicate guard)...")
    hashes = []
    for p in df_train["img_path"].astype(str).tolist():
        g = _read_gray_u8(p)
        hashes.append(phash64(g, hash_size=PHASH_SIZE))
    df_train["phash64"] = hashes

    # greedy grouping by hamming distance threshold
    # group assignment: for each sample, find earlier representative within dist, else new group
    reps = []  # list of (rep_hash, group_id)
    group_ids = [-1] * len(df_train)
    gid = 0
    for i, h in enumerate(df_train["phash64"].tolist()):
        assigned = False
        for rep_h, rep_gid in reps:
            if hamming64(h, rep_h) <= PHASH_MAX_DIST:
                group_ids[i] = rep_gid
                assigned = True
                break
        if not assigned:
            group_ids[i] = gid
            reps.append((h, gid))
            gid += 1
    df_train["group_id"] = np.array(group_ids, dtype=np.int32)
    print(f"Groups formed: {df_train['group_id'].nunique()} from {len(df_train)} images")
else:
    df_train["group_id"] = np.arange(len(df_train), dtype=np.int32)

# ----------------------------
# Buat folds "aman"
# Strategi:
# - Jika pHash grouping aktif: kita lakukan fold assignment pada level group_id,
#   dengan target stratify memakai proporsi forged per group.
# - Kalau tidak: stratify langsung per image.
# ----------------------------
df_train["fold"] = -1

if USE_PHASH_GROUP:
    gdf = (df_train
           .groupby("group_id", as_index=False)
           .agg(
               n=("case_id", "count"),
               y=("is_forged", "mean")  # mean forged fraction
           ))
    # bin y untuk stratify kasar (0 vs >0, dan beberapa level)
    # lebih stabil dibanding stratify float
    ybin = pd.cut(gdf["y"], bins=[-0.001, 0.001, 0.5, 1.0], labels=[0,1,2]).astype(int).to_numpy()

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold, (_, va_gidx) in enumerate(skf.split(gdf["group_id"].to_numpy(), ybin)):
        va_groups = set(gdf.iloc[va_gidx]["group_id"].tolist())
        df_train.loc[df_train["group_id"].isin(va_groups), "fold"] = fold
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    y = df_train["is_forged"].to_numpy()
    for fold, (_, va_idx) in enumerate(skf.split(df_train.index.to_numpy(), y)):
        df_train.loc[df_train.index[va_idx], "fold"] = fold

# ----------------------------
# Sanity check distribusi
# ----------------------------
print("\n===== FOLD CHECK =====")
for f in range(N_FOLDS):
    part = df_train[df_train["fold"].eq(f)]
    print(f"Fold {f}: n={len(part)} | forged%={part['is_forged'].mean():.4f} | groups={part['group_id'].nunique()}")

# ----------------------------
# Merge back ke df_index
# ----------------------------
df_index2 = df_index.copy()
df_index2["fold"] = -1
df_index2["group_id"] = -1

# mapping fold+group hanya untuk train
tmp = df_train[["case_id", "fold", "group_id"]].copy()
df_index2 = df_index2.merge(tmp, on="case_id", how="left", suffixes=("", "_y"))

# ambil hasil merge
df_index2["fold"] = df_index2["fold_y"].fillna(df_index2["fold"]).astype(int)
df_index2["group_id"] = df_index2["group_id_y"].fillna(df_index2["group_id"]).astype(int)
df_index2 = df_index2.drop(columns=[c for c in ["fold_y", "group_id_y"] if c in df_index2.columns])

df_index = df_index2
df_train = df_index[df_index["split"].eq("train")].reset_index(drop=True)
df_test  = df_index[df_index["split"].eq("test")].reset_index(drop=True)

# ----------------------------
# Save artifact
# ----------------------------
out_parq = os.path.join(PATHS["artifact_dir"], "df_index_with_folds.parquet")
out_csv  = os.path.join(PATHS["artifact_dir"], "df_index_with_folds.csv")

try:
    df_index.to_parquet(out_parq, index=False)
    print(f"\nSaved: {out_parq}")
except Exception as e:
    df_index.to_csv(out_csv, index=False)
    print(f"\nParquet gagal ({type(e).__name__}: {e}) -> Saved CSV: {out_csv}")


Building pHash groups (near-duplicate guard)...
Groups formed: 2632 from 2795 images

===== FOLD CHECK =====
Fold 0: n=558 | forged%=0.1505 | groups=527
Fold 1: n=559 | forged%=0.1503 | groups=527
Fold 2: n=553 | forged%=0.1501 | groups=526
Fold 3: n=568 | forged%=0.1461 | groups=526
Fold 4: n=557 | forged%=0.1508 | groups=526

Saved: /kaggle/working/recodai_luc/artifacts/df_index_with_folds.parquet


# Baseline nol: “semua authentic”

In [4]:
# ============================================================
# STAGE E0 — BASELINE NOL: SEMUA "authentic" (ONE CELL)
# Tujuan:
# - Membuat prediksi baseline paling sederhana: semua gambar dianggap authentic
# - Menghasilkan:
#   * df_oof_baseline: prediksi untuk train (OOF per fold) -> semua authentic
#   * submission_baseline.csv untuk test -> semua authentic
#
# Prasyarat:
# - PATHS, df_index, df_train, df_test sudah ada (Stage B/D)
# - sample_submission.csv ada
# ============================================================

import os
from pathlib import Path
import pandas as pd
import numpy as np

# ----------------------------
# Ensure df_index tersedia
# ----------------------------
if "df_index" not in globals() or not isinstance(df_index, pd.DataFrame):
    # load dari artifact folds jika ada, fallback ke df_index
    art = Path(PATHS["artifact_dir"]) / "df_index_with_folds.parquet"
    art_csv = Path(PATHS["artifact_dir"]) / "df_index_with_folds.csv"
    if art.exists():
        df_index = pd.read_parquet(art)
    elif art_csv.exists():
        df_index = pd.read_csv(art_csv)
    else:
        art2 = Path(PATHS["artifact_dir"]) / "df_index.parquet"
        art2_csv = Path(PATHS["artifact_dir"]) / "df_index.csv"
        if art2.exists():
            df_index = pd.read_parquet(art2)
        elif art2_csv.exists():
            df_index = pd.read_csv(art2_csv)
        else:
            raise RuntimeError("df_index tidak ada. Jalankan Stage B dulu.")

df_train = df_index[df_index["split"].eq("train")].reset_index(drop=True)
df_test  = df_index[df_index["split"].eq("test")].reset_index(drop=True)

# pastikan fold ada (kalau belum, set fold=-1)
if "fold" not in df_train.columns:
    df_train["fold"] = -1

# ----------------------------
# 1) OOF baseline untuk train
# ----------------------------
df_oof_baseline = df_train[["case_id", "fold"]].copy()
df_oof_baseline["annotation_pred"] = "authentic"

print("===== BASELINE E0 (ALL AUTHENTIC) =====")
print(f"OOF rows: {len(df_oof_baseline)}")
print(df_oof_baseline.head())

# ----------------------------
# 2) Submission baseline untuk test (ikuti sample_submission)
# ----------------------------
sample_path = PATHS["sample_submission"]
if not Path(sample_path).exists():
    raise FileNotFoundError(f"sample_submission.csv tidak ditemukan: {sample_path}")

sub = pd.read_csv(sample_path)
if "case_id" not in sub.columns or "annotation" not in sub.columns:
    raise ValueError(f"sample_submission harus punya kolom case_id & annotation. Kolom ada: {list(sub.columns)}")

sub["case_id"] = sub["case_id"].astype(str)
sub["annotation"] = "authentic"

out_dir = "/kaggle/working/recodai_luc/outputs"
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "submission_baseline_all_authentic.csv")
sub.to_csv(out_path, index=False)

print("\nSaved baseline submission:")
print(out_path)
print(sub.head())


===== BASELINE E0 (ALL AUTHENTIC) =====
OOF rows: 2795
  case_id  fold annotation_pred
0      10     1       authentic
1   10015     0       authentic
2   10017     1       authentic
3   10030     2       authentic
4   10070     4       authentic

Saved baseline submission:
/kaggle/working/recodai_luc/outputs/submission_baseline_all_authentic.csv
  case_id annotation
0      45  authentic


# Baseline CPU cepat: detektor copy-move klasik (tanpa DINO)

In [5]:
# ============================================================
# STAGE E1 — BASELINE CPU CEPAT: DETEKTOR COPY-MOVE KLASIK (TANPA DINO) — ONE CELL
# Tujuan:
# - Membuat baseline yang lebih kuat dari "all authentic" dengan CV klasik:
#   ORB self-match + offset consensus + fallback patch-hash self-similarity
#
# Prasyarat:
# - PATHS, df_index, df_train, df_test sudah ada (Stage B/D)
# - sample_submission.csv ada
#
# Output:
# - artifacts/pred_classic.csv          (pred train+test: annotation_pred + confidence + area_frac)
# - outputs/submission_baseline_classic.csv
# - df_pred_classic (DataFrame) di memori
# ============================================================

import os, re
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

# ----------------------------
# Ensure df_index tersedia
# ----------------------------
if "df_index" not in globals() or not isinstance(df_index, pd.DataFrame):
    art = Path(PATHS["artifact_dir"]) / "df_index_with_folds.parquet"
    art_csv = Path(PATHS["artifact_dir"]) / "df_index_with_folds.csv"
    if art.exists():
        df_index = pd.read_parquet(art)
    elif art_csv.exists():
        df_index = pd.read_csv(art_csv)
    else:
        art2 = Path(PATHS["artifact_dir"]) / "df_index.parquet"
        art2_csv = Path(PATHS["artifact_dir"]) / "df_index.csv"
        if art2.exists():
            df_index = pd.read_parquet(art2)
        elif art2_csv.exists():
            df_index = pd.read_csv(art2_csv)
        else:
            raise RuntimeError("df_index tidak ada. Jalankan Stage B dulu.")

df_train = df_index[df_index["split"].eq("train")].reset_index(drop=True)
df_test  = df_index[df_index["split"].eq("test")].reset_index(drop=True)

# ----------------------------
# OUTPUT DIRS
# ----------------------------
work_root = PATHS.get("work_root", "/kaggle/working/recodai_luc")
out_dir = os.path.join(work_root, "outputs")
art_dir = PATHS.get("artifact_dir", os.path.join(work_root, "artifacts"))
os.makedirs(out_dir, exist_ok=True)
os.makedirs(art_dir, exist_ok=True)

# ----------------------------
# CONFIG (tuning cepat di CPU)
# ----------------------------
CFG_CLASSIC = {
    # resize untuk deteksi (lebih kecil = lebih cepat)
    "max_side": 640,

    # ORB
    "orb_nfeatures": 5000,
    "orb_ratio": 0.75,
    "orb_min_spatial_sep": 18,    # px
    "orb_bin_px": 4,
    "orb_min_bin_count": 18,
    "orb_top_bins": 2,
    "kp_radius": 10,

    # Patch-hash fallback
    "ph_sizes": (16, 24),
    "ph_stride_frac": 0.5,
    "ph_min_std": 8.0,
    "ph_bin_px": 4,
    "ph_min_bin_count": 14,
    "ph_top_bins": 2,
    "ph_max_bucket": 8,
    "ph_ncc_thr": 0.93,

    # Postprocess
    "close_ks": 7,
    "dilate_ks": 5,
    "min_area_frac": 0.00035,     # buang komponen kecil
    "auth_area_frac": 0.00045,    # jika total area < ini -> authentic
    "auth_conf_thr": 16,          # jika confidence < ini -> authentic (guard noise)
}

IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# ----------------------------
# UTILS
# ----------------------------
def imread_any(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return None
    if img.ndim == 2:
        return img
    if img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    return img

def to_gray_u8(img):
    if img.ndim == 2:
        g = img
    else:
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if g.dtype != np.uint8:
        g = cv2.normalize(g, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return g

def resize_keep(gray_u8, max_side):
    h, w = gray_u8.shape[:2]
    s = max(h, w)
    if s <= max_side:
        return gray_u8, 1.0
    scale = max_side / float(s)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    out = cv2.resize(gray_u8, (nw, nh), interpolation=cv2.INTER_AREA)
    return out, scale

def rle_encode(mask_u8):
    # mask_u8: HxW {0,1}
    pixels = mask_u8.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    if len(runs) == 0:
        return ""
    return " ".join(str(x) for x in runs)

def morph_post(mask, close_ks=7, dilate_ks=5):
    mask = (mask > 0).astype(np.uint8)
    if close_ks and close_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_ks, close_ks))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=1)
    if dilate_ks and dilate_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_ks, dilate_ks))
        mask = cv2.dilate(mask, k, iterations=1)
    return mask

def remove_small_components(mask, min_area):
    num, lab, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = np.zeros_like(mask)
    for i in range(1, num):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= min_area:
            out[lab == i] = 1
    return out

# ----------------------------
# ORB copy-move (self match + offset consensus)
# ----------------------------
def orb_copymove_mask(gray_u8, cfg):
    orb = cv2.ORB_create(
        nfeatures=int(cfg["orb_nfeatures"]),
        scaleFactor=1.2,
        nlevels=8,
        edgeThreshold=15,
        patchSize=31,
        fastThreshold=7
    )
    kps, des = orb.detectAndCompute(gray_u8, None)
    if des is None or len(kps) < 80:
        return None, 0

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    knn = bf.knnMatch(des, des, k=2)

    pts1, pts2 = [], []
    for pair in knn:
        if len(pair) < 2:
            continue
        m, n = pair[0], pair[1]
        if m.queryIdx == m.trainIdx:
            continue
        if m.distance >= cfg["orb_ratio"] * n.distance:
            continue
        p1 = np.array(kps[m.queryIdx].pt, dtype=np.float32)
        p2 = np.array(kps[m.trainIdx].pt, dtype=np.float32)
        if np.linalg.norm(p2 - p1) < cfg["orb_min_spatial_sep"]:
            continue
        pts1.append(p1); pts2.append(p2)

    if len(pts1) < cfg["orb_min_bin_count"]:
        return None, 0

    pts1 = np.stack(pts1, 0)
    pts2 = np.stack(pts2, 0)
    dxy = pts2 - pts1

    b = float(cfg["orb_bin_px"])
    bins = np.round(dxy / b).astype(np.int32)
    key = bins[:,0].astype(np.int64) * 1000003 + bins[:,1].astype(np.int64)

    uniq, cnt = np.unique(key, return_counts=True)
    order = np.argsort(-cnt)

    H, W = gray_u8.shape[:2]
    mask = np.zeros((H, W), np.uint8)

    best_cnt = int(cnt[order[0]]) if len(order) else 0
    used_bins = 0

    for idx in order[: int(cfg["orb_top_bins"]) ]:
        if cnt[idx] < cfg["orb_min_bin_count"]:
            break
        k = uniq[idx]
        sel = (key == k)
        p1s = pts1[sel]
        p2s = pts2[sel]
        used_bins += 1
        for p in np.vstack([p1s, p2s]):
            cv2.circle(mask, (int(p[0]), int(p[1])), int(cfg["kp_radius"]), 1, -1)

    if used_bins == 0:
        return None, 0

    mask = morph_post(mask, close_ks=cfg["close_ks"], dilate_ks=cfg["dilate_ks"])
    return mask, best_cnt

# ----------------------------
# Patch-hash fallback
# ----------------------------
def ahash64(patch_u8):
    small = cv2.resize(patch_u8, (8, 8), interpolation=cv2.INTER_AREA).astype(np.float32)
    m = small.mean()
    bits = (small > m).astype(np.uint8).flatten()
    v = 0
    for b in bits:
        v = (v << 1) | int(b)
    return v

def ncc(a, b):
    a = a.astype(np.float32); b = b.astype(np.float32)
    a -= a.mean(); b -= b.mean()
    da = np.sqrt((a*a).sum()) + 1e-6
    db = np.sqrt((b*b).sum()) + 1e-6
    return float((a*b).sum() / (da*db))

def patchhash_copymove_mask(gray_u8, cfg):
    H, W = gray_u8.shape[:2]
    acc = np.zeros((H, W), np.uint8)
    best_cnt_all = 0

    for psz in cfg["ph_sizes"]:
        stride = max(4, int(round(psz * float(cfg["ph_stride_frac"]))))

        buckets = {}
        for y in range(0, H - psz + 1, stride):
            for x in range(0, W - psz + 1, stride):
                patch = gray_u8[y:y+psz, x:x+psz]
                if patch.std() < cfg["ph_min_std"]:
                    continue
                h = ahash64(patch)
                buckets.setdefault(h, []).append((x, y))

        offsets = []
        pairs = []
        for h, pos in buckets.items():
            if len(pos) < 2 or len(pos) > cfg["ph_max_bucket"]:
                continue
            for i in range(len(pos)):
                for j in range(i+1, len(pos)):
                    (x1,y1) = pos[i]; (x2,y2) = pos[j]
                    if abs(x2-x1) + abs(y2-y1) < psz:
                        continue
                    p1 = gray_u8[y1:y1+psz, x1:x1+psz]
                    p2 = gray_u8[y2:y2+psz, x2:x2+psz]
                    if ncc(p1, p2) >= cfg["ph_ncc_thr"]:
                        offsets.append((x2-x1, y2-y1))
                        pairs.append(((x1,y1),(x2,y2)))

        if len(offsets) < cfg["ph_min_bin_count"]:
            continue

        offsets = np.array(offsets, dtype=np.float32)
        b = float(cfg["ph_bin_px"])
        bins = np.round(offsets / b).astype(np.int32)
        key = bins[:,0].astype(np.int64) * 1000003 + bins[:,1].astype(np.int64)

        uniq, cnt = np.unique(key, return_counts=True)
        order = np.argsort(-cnt)

        best_cnt = int(cnt[order[0]]) if len(order) else 0
        best_cnt_all = max(best_cnt_all, best_cnt)

        mask = np.zeros((H, W), np.uint8)
        used_bins = 0
        for idx in order[: int(cfg["ph_top_bins"]) ]:
            if cnt[idx] < cfg["ph_min_bin_count"]:
                break
            k = uniq[idx]
            sel = (key == k)
            used_bins += 1
            for t, ok in enumerate(sel):
                if not ok:
                    continue
                (x1,y1),(x2,y2) = pairs[t]
                mask[y1:y1+psz, x1:x1+psz] = 1
                mask[y2:y2+psz, x2:x2+psz] = 1

        if used_bins > 0:
            mask = morph_post(mask, close_ks=cfg["close_ks"], dilate_ks=cfg["dilate_ks"])
            acc = np.maximum(acc, mask)

    if acc.sum() == 0:
        return None, 0
    return acc, best_cnt_all

# ----------------------------
# Main detect
# ----------------------------
def detect_classic_mask(img, cfg):
    gray = to_gray_u8(img)
    small, scale = resize_keep(gray, cfg["max_side"])

    m1, c1 = orb_copymove_mask(small, cfg)
    m2, c2 = patchhash_copymove_mask(small, cfg)

    if m1 is None and m2 is None:
        H0, W0 = gray.shape[:2]
        return np.zeros((H0, W0), np.uint8), 0, 0.0

    m = m1 if m2 is None else (m2 if m1 is None else np.maximum(m1, m2))
    conf = int(c1 + c2)

    if scale != 1.0:
        H0, W0 = gray.shape[:2]
        m = cv2.resize(m, (W0, H0), interpolation=cv2.INTER_NEAREST)

    m = (m > 0).astype(np.uint8)
    m = morph_post(m, close_ks=cfg["close_ks"], dilate_ks=cfg["dilate_ks"])

    H, W = m.shape[:2]
    min_area = int(cfg["min_area_frac"] * H * W)
    if min_area > 0:
        m = remove_small_components(m, min_area=min_area)

    area_frac = float(m.mean())
    return m, conf, area_frac

# ----------------------------
# Build mapping case_id -> path (train+test)
# ----------------------------
cid2path = dict(zip(df_index["case_id"].astype(str), df_index["img_path"].astype(str)))

# ----------------------------
# Predict for train+test (store as table; tidak simpan mask besar)
# ----------------------------
rows = []

all_case_ids = df_index["case_id"].astype(str).tolist()
for cid in tqdm(all_case_ids, desc="Classic baseline predict (train+test)"):
    p = cid2path.get(cid, None)
    if p is None or (not Path(p).exists()):
        rows.append({
            "case_id": cid,
            "split": df_index.loc[df_index["case_id"].astype(str).eq(cid), "split"].iloc[0],
            "annotation_pred": "authentic",
            "confidence": 0,
            "area_frac": 0.0,
        })
        continue

    img = imread_any(p)
    if img is None:
        rows.append({
            "case_id": cid,
            "split": df_index.loc[df_index["case_id"].astype(str).eq(cid), "split"].iloc[0],
            "annotation_pred": "authentic",
            "confidence": 0,
            "area_frac": 0.0,
        })
        continue

    mask, conf, area_frac = detect_classic_mask(img, CFG_CLASSIC)

    # decision: authentic vs rle
    if (area_frac < CFG_CLASSIC["auth_area_frac"]) or (conf < CFG_CLASSIC["auth_conf_thr"]):
        anno = "authentic"
    else:
        rle = rle_encode(mask)
        anno = rle if len(rle) else "authentic"

    rows.append({
        "case_id": cid,
        "split": df_index.loc[df_index["case_id"].astype(str).eq(cid), "split"].iloc[0],
        "annotation_pred": anno,
        "confidence": int(conf),
        "area_frac": float(area_frac),
    })

df_pred_classic = pd.DataFrame(rows)

# Merge fold info (jika ada)
if "fold" in df_index.columns:
    df_pred_classic = df_pred_classic.merge(df_index[["case_id", "fold"]], on="case_id", how="left")

# Save predictions artifact
pred_path = os.path.join(art_dir, "pred_classic.csv")
df_pred_classic.to_csv(pred_path, index=False)
print(f"\nSaved: {pred_path}")
print(df_pred_classic.head())

# ----------------------------
# Build submission for test (urut sesuai sample_submission)
# ----------------------------
sample_path = PATHS["sample_submission"]
sub = pd.read_csv(sample_path)
sub["case_id"] = sub["case_id"].astype(str)

pred_map = dict(zip(df_pred_classic.loc[df_pred_classic["split"].eq("test"), "case_id"],
                    df_pred_classic.loc[df_pred_classic["split"].eq("test"), "annotation_pred"]))

sub["annotation"] = sub["case_id"].map(lambda x: pred_map.get(str(x), "authentic"))
out_sub = os.path.join(out_dir, "submission_baseline_classic.csv")
sub.to_csv(out_sub, index=False)

print(f"\nSaved submission: {out_sub}")
print(sub.head())


Classic baseline predict (train+test):   0%|          | 0/2796 [00:00<?, ?it/s]


Saved: /kaggle/working/recodai_luc/artifacts/pred_classic.csv
  case_id  split                                    annotation_pred  \
0      10  train                                          authentic   
1   10015  train                                          authentic   
2   10017  train                                          authentic   
3   10030  train  22521 43 23187 43 23852 45 24517 46 25183 46 2...   
4   10070  train                                          authentic   

   confidence  area_frac  fold  
0           0   0.000000     1  
1           0   0.000000     0  
2           0   0.000000     1  
3          31   0.128553     2  
4           0   0.000000     4  

Saved submission: /kaggle/working/recodai_luc/outputs/submission_baseline_classic.csv
  case_id                                         annotation
0      45  170590 84 170791 84 172034 84 172235 84 173477...


# DINOv2 feature extraction

In [6]:
# ============================================================
# STAGE E — DINOv2 FEATURE EXTRACTION (CPU, CACHE KE DISK) — ONE CELL
# Model path (sesuai revisi kamu):
#   /kaggle/input/dinov2/pytorch/small/1
#
# Prasyarat:
# - PATHS["dinov2_model_dir"] ada (Stage B revisi)
# - df_index sudah ada (Stage B/D)
#
# Output:
# - processor, dinov2_model (eval, CPU)
# - CFG_DINO (param ekstraksi)
# - get_feat(case_id) -> dict (feat, gh, gw, scale, H0, W0, H1, W1, Hp, Wp, pad_h, pad_w, patch_size)
# - Cache file:
#   /kaggle/working/recodai_luc/cache/feat_dino/{case_id}.npz
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

import torch
from transformers import AutoImageProcessor, AutoModel

# ----------------------------
# Ensure df_index tersedia
# ----------------------------
if "df_index" not in globals() or not isinstance(df_index, pd.DataFrame):
    art = Path(PATHS["artifact_dir"]) / "df_index_with_folds.parquet"
    art_csv = Path(PATHS["artifact_dir"]) / "df_index_with_folds.csv"
    if art.exists():
        df_index = pd.read_parquet(art)
    elif art_csv.exists():
        df_index = pd.read_csv(art_csv)
    else:
        art2 = Path(PATHS["artifact_dir"]) / "df_index.parquet"
        art2_csv = Path(PATHS["artifact_dir"]) / "df_index.csv"
        if art2.exists():
            df_index = pd.read_parquet(art2)
        elif art2_csv.exists():
            df_index = pd.read_csv(art2_csv)
        else:
            raise RuntimeError("df_index tidak ada. Jalankan Stage B dulu.")

# ----------------------------
# Paths & cache dirs
# ----------------------------
work_root = PATHS.get("work_root", "/kaggle/working/recodai_luc")
cache_dir = os.path.join(work_root, "cache", "feat_dino")
os.makedirs(cache_dir, exist_ok=True)

DINO_DIR = PATHS.get("dinov2_model_dir", "/kaggle/input/dinov2/pytorch/small/1")
if not Path(DINO_DIR).exists():
    raise FileNotFoundError(f"DINOv2 dir tidak ditemukan: {DINO_DIR}")

# ----------------------------
# Config ekstraksi
# ----------------------------
CFG_DINO = {
    "max_side": 560,        # CPU friendly; bisa 448/560/672 (makin besar makin lambat)
    "patch_size": 14,       # DINOv2 ViT-S/14
    "use_fp16_store": True, # simpan feat float16 agar hemat disk
    "skip_if_cached": True,
    "min_side": 224,        # kalau gambar kecil, tetap minimal segini (opsional)
}

# ----------------------------
# Load processor + model (local)
# ----------------------------
processor = AutoImageProcessor.from_pretrained(DINO_DIR, local_files_only=True)
dinov2_model = AutoModel.from_pretrained(DINO_DIR, local_files_only=True)
dinov2_model.eval()
dinov2_model.to("cpu")

# mean/std untuk normalisasi manual (robust)
IMG_MEAN = np.array(getattr(processor, "image_mean", [0.485, 0.456, 0.406]), dtype=np.float32)
IMG_STD  = np.array(getattr(processor, "image_std",  [0.229, 0.224, 0.225]), dtype=np.float32)

# ----------------------------
# Utilities
# ----------------------------
IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def _imread_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return None
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def _resize_keep_rgb(img_rgb, max_side, min_side=0):
    H0, W0 = img_rgb.shape[:2]
    s0 = max(H0, W0)

    # scale down jika besar
    scale = 1.0
    if s0 > max_side:
        scale = max_side / float(s0)
    # scale up jika terlalu kecil (optional)
    if min_side and min(H0, W0) * scale < min_side:
        scale = max(scale, min_side / float(min(H0, W0)))

    if abs(scale - 1.0) < 1e-6:
        return img_rgb, 1.0

    H1 = int(round(H0 * scale))
    W1 = int(round(W0 * scale))
    img2 = cv2.resize(img_rgb, (W1, H1), interpolation=cv2.INTER_AREA if scale < 1.0 else cv2.INTER_CUBIC)
    return img2, scale

def _pad_to_multiple(img_rgb, patch):
    H, W = img_rgb.shape[:2]
    Hp = int(np.ceil(H / patch) * patch)
    Wp = int(np.ceil(W / patch) * patch)
    pad_h = Hp - H
    pad_w = Wp - W
    if pad_h == 0 and pad_w == 0:
        return img_rgb, (0, 0), (H, W), (Hp, Wp)
    # pad bottom/right dengan reflect agar tidak bikin tepi aneh
    img_pad = cv2.copyMakeBorder(img_rgb, 0, pad_h, 0, pad_w, borderType=cv2.BORDER_REFLECT_101)
    return img_pad, (pad_h, pad_w), (H, W), (Hp, Wp)

def _to_tensor_normalized(img_rgb):
    # img_rgb uint8 -> float32 [0,1] -> normalize -> torch tensor [1,3,H,W]
    x = img_rgb.astype(np.float32) / 255.0
    x = (x - IMG_MEAN[None, None, :]) / IMG_STD[None, None, :]
    x = np.transpose(x, (2, 0, 1))  # CHW
    return torch.from_numpy(x).unsqueeze(0)  # BCHW

def _feat_path(case_id):
    return os.path.join(cache_dir, f"{case_id}.npz")

# ----------------------------
# Feature extraction per image
# ----------------------------
@torch.no_grad()
def extract_and_cache_one(case_id, img_path):
    cid = str(case_id)
    outp = _feat_path(cid)
    if CFG_DINO["skip_if_cached"] and Path(outp).exists():
        return outp, True

    img = _imread_rgb(img_path)
    if img is None:
        raise FileNotFoundError(f"Gagal baca image: {img_path}")

    H0, W0 = img.shape[:2]

    img_rs, scale = _resize_keep_rgb(img, CFG_DINO["max_side"], min_side=CFG_DINO["min_side"])
    H1, W1 = img_rs.shape[:2]

    img_pad, (pad_h, pad_w), (H1a, W1a), (Hp, Wp) = _pad_to_multiple(img_rs, CFG_DINO["patch_size"])

    x = _to_tensor_normalized(img_pad)  # [1,3,Hp,Wp]
    x = x.to("cpu")

    out = dinov2_model(pixel_values=x)
    # last_hidden_state: [1, 1+gh*gw, dim] (ada CLS token)
    hs = out.last_hidden_state  # torch
    hs = hs[0]                  # [seq, dim]
    if hs.ndim != 2 or hs.shape[0] < 2:
        raise RuntimeError(f"Unexpected hidden_state shape: {tuple(hs.shape)} for case_id={cid}")

    patch_tokens = hs[1:, :]  # buang CLS
    P, D = patch_tokens.shape

    gh = Hp // CFG_DINO["patch_size"]
    gw = Wp // CFG_DINO["patch_size"]
    if gh * gw != P:
        # fallback: infer dari P (bila model/processor melakukan hal berbeda)
        # coba cari faktor mendekati aspect ratio
        ar = Wp / float(Hp)
        gw2 = int(round(np.sqrt(P * ar)))
        gw2 = max(1, min(P, gw2))
        gh2 = P // gw2
        if gh2 * gw2 != P:
            # last fallback: 1 x P
            gh2, gw2 = 1, P
        gh, gw = gh2, gw2

    feat = patch_tokens.detach().cpu().numpy()
    if CFG_DINO["use_fp16_store"]:
        feat = feat.astype(np.float16)

    np.savez_compressed(
        outp,
        feat=feat,
        gh=np.int32(gh),
        gw=np.int32(gw),
        patch_size=np.int32(CFG_DINO["patch_size"]),
        scale=np.float32(scale),
        H0=np.int32(H0), W0=np.int32(W0),
        H1=np.int32(H1), W1=np.int32(W1),
        Hp=np.int32(Hp), Wp=np.int32(Wp),
        pad_h=np.int32(pad_h), pad_w=np.int32(pad_w),
        img_path=str(img_path),
    )
    return outp, False

def get_feat(case_id):
    cid = str(case_id)
    p = _feat_path(cid)
    if not Path(p).exists():
        # coba ekstrak on-demand
        row = df_index[df_index["case_id"].astype(str).eq(cid)]
        if len(row) == 0:
            raise KeyError(f"case_id tidak ditemukan di df_index: {cid}")
        img_path = row["img_path"].iloc[0]
        extract_and_cache_one(cid, img_path)

    z = np.load(p, allow_pickle=True)
    return {
        "feat": z["feat"],  # [P,D]
        "gh": int(z["gh"]),
        "gw": int(z["gw"]),
        "patch_size": int(z["patch_size"]),
        "scale": float(z["scale"]),
        "H0": int(z["H0"]), "W0": int(z["W0"]),
        "H1": int(z["H1"]), "W1": int(z["W1"]),
        "Hp": int(z["Hp"]), "Wp": int(z["Wp"]),
        "pad_h": int(z["pad_h"]), "pad_w": int(z["pad_w"]),
        "path": str(z["img_path"]),
        "npz_path": p,
    }

# ----------------------------
# Run extraction (train+test) dengan skip cached
# ----------------------------
all_rows = df_index[["case_id", "img_path", "split"]].copy()
all_rows["case_id"] = all_rows["case_id"].astype(str)

done = 0
cached = 0
errors = 0

print("===== STAGE E: DINOv2 FEATURE EXTRACTION =====")
print("Model dir:", DINO_DIR)
print("Cache dir:", cache_dir)
print("Total images:", len(all_rows))

for cid, img_path, split in tqdm(all_rows.itertuples(index=False), total=len(all_rows), desc="Extract DINO feats"):
    try:
        outp, was_cached = extract_and_cache_one(cid, img_path)
        done += 1
        cached += int(was_cached)
    except Exception as e:
        errors += 1
        # jangan stop total; catat 1-2 contoh
        if errors <= 5:
            print(f"[ERROR] case_id={cid} split={split} | {type(e).__name__}: {e}")

print("\n===== STAGE E DONE =====")
print(f"Processed: {done} | cached: {cached} | errors: {errors}")

# Quick sanity check: load 1 item
if len(all_rows) > 0:
    cid0 = all_rows["case_id"].iloc[0]
    ex = get_feat(cid0)
    print("\nExample feature:")
    print("case_id:", cid0)
    print("feat shape:", ex["feat"].shape, "| gh,gw:", ex["gh"], ex["gw"], "| orig:", (ex["H0"], ex["W0"]), "| resized:", (ex["H1"], ex["W1"]), "| pad:", (ex["Hp"], ex["Wp"]))


2025-12-30 19:58:22.520418: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767124702.751133      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767124702.819206      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767124703.404379      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767124703.404428      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767124703.404430      17 computation_placer.cc:177] computation placer alr

===== STAGE E: DINOv2 FEATURE EXTRACTION =====
Model dir: /kaggle/input/dinov2/pytorch/small/1
Cache dir: /kaggle/working/recodai_luc/cache/feat_dino
Total images: 2796


Extract DINO feats:   0%|          | 0/2796 [00:00<?, ?it/s]


===== STAGE E DONE =====
Processed: 2796 | cached: 0 | errors: 0

Example feature:
case_id: 10
feat shape: (1280, 384) | gh,gw: 32 40 | orig: (512, 648) | resized: (442, 560) | pad: (448, 560)


# DINO self-similarity → pasangan patch → “offset consensus”

In [7]:
# ============================================================
# STAGE F — DINO SELF-SIMILARITY -> PASANGAN PATCH -> "OFFSET CONSENSUS" (ONE CELL)
# Tujuan:
# - Dari fitur DINOv2 (cache Stage E), buat mask kandidat duplikasi berbasis self-similarity
# - Mekanisme:
#   1) cosine similarity antar patch (P x P)
#   2) untuk tiap patch: ambil pasangan match terbaik yang "jauh" (hindari tetangga dekat)
#   3) hitung offset (dx,dy) patch -> voting/binning -> ambil cluster offset terbesar
#   4) patch yang konsisten di cluster => mask grid -> upsample -> mask pixel (H0,W0)
#
# Prasyarat:
# - PATHS, df_index sudah ada
# - get_feat(case_id) sudah ada (Stage E)
#
# Output:
# - cache mask:
#   /kaggle/working/recodai_luc/cache/pred_dino_raw/{case_id}.npz
#   berisi: mask_u8 (H0,W0), confidence, area_frac
# - df_pred_dino_raw (train+test): case_id, split, confidence, area_frac, npz_path
# - helper: get_pred_dino_raw(case_id)
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

# ----------------------------
# Ensure df_index tersedia
# ----------------------------
if "df_index" not in globals() or not isinstance(df_index, pd.DataFrame):
    art = Path(PATHS["artifact_dir"]) / "df_index_with_folds.parquet"
    art_csv = Path(PATHS["artifact_dir"]) / "df_index_with_folds.csv"
    if art.exists():
        df_index = pd.read_parquet(art)
    elif art_csv.exists():
        df_index = pd.read_csv(art_csv)
    else:
        art2 = Path(PATHS["artifact_dir"]) / "df_index.parquet"
        art2_csv = Path(PATHS["artifact_dir"]) / "df_index.csv"
        if art2.exists():
            df_index = pd.read_parquet(art2)
        elif art2_csv.exists():
            df_index = pd.read_csv(art2_csv)
        else:
            raise RuntimeError("df_index tidak ada. Jalankan Stage B/D dulu.")

# pastikan get_feat ada
if "get_feat" not in globals() or not callable(get_feat):
    raise RuntimeError("get_feat(case_id) tidak ada. Jalankan Stage E (DINOv2 feature extraction) dulu.")

# ----------------------------
# Cache dirs
# ----------------------------
work_root = PATHS.get("work_root", "/kaggle/working/recodai_luc")
pred_dir = os.path.join(work_root, "cache", "pred_dino_raw")
os.makedirs(pred_dir, exist_ok=True)

# ----------------------------
# CONFIG self-similarity
# ----------------------------
CFG_SIM = {
    "topk": 20,                 # kandidat nearest neighbor per patch
    "min_sep_patches": 3,        # minimal jarak (Manhattan) antar patch untuk dianggap duplikasi
    "sim_thr": 0.80,            # threshold cosine similarity minimal untuk dipakai voting
    "bin_dxdy": 1,              # binning offset dalam satuan patch
    "min_bin_count": 14,        # minimal voting di 1 bin offset agar dianggap duplikasi nyata
    "top_bins": 2,              # ambil beberapa cluster offset terbaik
    "close_ks": 0,              # postprocess ringan (0 = off)
    "dilate_ks": 0,             # postprocess ringan (0 = off)
    "min_area_frac": 0.00020,   # buang komponen kecil (opsional)
}

# ----------------------------
# Helpers
# ----------------------------
def _pred_npz_path(case_id):
    return os.path.join(pred_dir, f"{case_id}.npz")

def _morph_post(mask_u8, close_ks=0, dilate_ks=0):
    m = (mask_u8 > 0).astype(np.uint8)
    if close_ks and close_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_ks, close_ks))
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k, iterations=1)
    if dilate_ks and dilate_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_ks, dilate_ks))
        m = cv2.dilate(m, k, iterations=1)
    return m

def _remove_small_components(mask, min_area):
    if min_area <= 0:
        return mask
    num, lab, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = np.zeros_like(mask)
    for i in range(1, num):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area >= min_area:
            out[lab == i] = 1
    return out

def _coords_from_index(idx, gw):
    y = idx // gw
    x = idx - y * gw
    return int(x), int(y)

def _manhattan_far(ix, iy, jx, jy, min_sep):
    return (abs(ix - jx) + abs(iy - jy)) >= min_sep

def _norm_feats(feat):
    # feat: (P,D) float16/float32
    f = feat.astype(np.float32, copy=False)
    n = np.linalg.norm(f, axis=1, keepdims=True) + 1e-6
    return f / n

def dino_selfsim_mask_from_feat(feat_dict, cfg):
    feat = feat_dict["feat"]   # [P,D]
    gh = int(feat_dict["gh"]); gw = int(feat_dict["gw"])
    patch = int(feat_dict["patch_size"])
    H0 = int(feat_dict["H0"]); W0 = int(feat_dict["W0"])
    H1 = int(feat_dict["H1"]); W1 = int(feat_dict["W1"])
    Hp = int(feat_dict["Hp"]); Wp = int(feat_dict["Wp"])

    P = int(feat.shape[0])
    if P <= 8 or gh * gw != P:
        # fallback aman
        return np.zeros((H0, W0), np.uint8), 0, 0.0

    # 1) normalize
    f = _norm_feats(feat)  # (P,D) float32

    # 2) similarity matrix (cosine)
    #    (P,P) float32
    sim = f @ f.T
    np.fill_diagonal(sim, -1.0)  # exclude self

    # precompute coords
    xs = (np.arange(P) % gw).astype(np.int32)
    ys = (np.arange(P) // gw).astype(np.int32)

    topk = int(cfg["topk"])
    min_sep = int(cfg["min_sep_patches"])
    sim_thr = float(cfg["sim_thr"])

    # store chosen matches
    match_j = np.full(P, -1, dtype=np.int32)
    match_s = np.full(P, -1.0, dtype=np.float32)

    # 3) choose best far match per patch using argpartition topk
    #    (lebih cepat daripada sort penuh)
    for i in range(P):
        row = sim[i]
        if topk >= P:
            cand = np.argsort(-row)
        else:
            idx = np.argpartition(-row, topk)[:topk]
            # urutkan kecil agar ambil yang terbaik dulu
            idx = idx[np.argsort(-row[idx])]
            cand = idx

        ix, iy = xs[i], ys[i]
        chosen = -1
        chosen_s = -1.0
        for j in cand:
            s = float(row[j])
            if s < sim_thr:
                break
            jx, jy = xs[j], ys[j]
            if not _manhattan_far(ix, iy, jx, jy, min_sep):
                continue
            chosen = int(j)
            chosen_s = s
            break

        if chosen >= 0:
            match_j[i] = chosen
            match_s[i] = chosen_s

    valid = match_j >= 0
    if valid.sum() < cfg["min_bin_count"]:
        return np.zeros((H0, W0), np.uint8), 0, 0.0

    # 4) offset voting/binning
    dx = (xs[match_j[valid]] - xs[valid]).astype(np.int32)
    dy = (ys[match_j[valid]] - ys[valid]).astype(np.int32)

    b = int(cfg["bin_dxdy"])
    dxq = np.round(dx / b).astype(np.int32)
    dyq = np.round(dy / b).astype(np.int32)

    key = dxq.astype(np.int64) * 1000003 + dyq.astype(np.int64)
    uniq, cnt = np.unique(key, return_counts=True)
    order = np.argsort(-cnt)

    # pick top bins
    picked_keys = []
    for idx in order[: int(cfg["top_bins"]) ]:
        if int(cnt[idx]) < int(cfg["min_bin_count"]):
            break
        picked_keys.append(int(uniq[idx]))

    if not picked_keys:
        return np.zeros((H0, W0), np.uint8), 0, 0.0

    # confidence = count top bin
    conf = int(cnt[order[0]])

    # 5) build mask grid from patches in selected bins
    mask_grid = np.zeros((gh, gw), dtype=np.uint8)

    valid_idx = np.where(valid)[0]
    for k in picked_keys:
        sel = (key == k)
        ii = valid_idx[sel]
        jj = match_j[valid][sel]

        # mark both source and matched patches
        mask_grid[ys[ii], xs[ii]] = 1
        mask_grid[ys[jj], xs[jj]] = 1

    # 6) upsample grid -> pixel mask (Hp,Wp) then crop pad -> resize to original
    mask_pad = cv2.resize(mask_grid, (Wp, Hp), interpolation=cv2.INTER_NEAREST).astype(np.uint8)
    # crop to resized (H1,W1) excluding pad bottom/right
    mask_rs = mask_pad[:H1, :W1]
    # resize to original
    mask0 = cv2.resize(mask_rs, (W0, H0), interpolation=cv2.INTER_NEAREST).astype(np.uint8)

    # optional postprocess + remove small components
    mask0 = _morph_post(mask0, close_ks=cfg["close_ks"], dilate_ks=cfg["dilate_ks"])
    min_area = int(float(cfg["min_area_frac"]) * H0 * W0)
    mask0 = _remove_small_components(mask0, min_area=min_area)

    area_frac = float(mask0.mean())
    return mask0, conf, area_frac

def save_pred_dino_raw(case_id, mask_u8, conf, area_frac):
    cid = str(case_id)
    outp = _pred_npz_path(cid)
    np.savez_compressed(outp, mask_u8=mask_u8.astype(np.uint8), confidence=np.int32(conf), area_frac=np.float32(area_frac))
    return outp

def get_pred_dino_raw(case_id):
    cid = str(case_id)
    p = _pred_npz_path(cid)
    if not Path(p).exists():
        raise FileNotFoundError(f"Pred cache tidak ditemukan untuk {cid}: {p}")
    z = np.load(p, allow_pickle=True)
    return {
        "mask_u8": z["mask_u8"].astype(np.uint8),
        "confidence": int(z["confidence"]),
        "area_frac": float(z["area_frac"]),
        "npz_path": p
    }

# ----------------------------
# Run for all images (train+test), skip cached
# ----------------------------
rows = []
all_rows = df_index[["case_id", "img_path", "split"]].copy()
all_rows["case_id"] = all_rows["case_id"].astype(str)

processed = 0
cached = 0
errors = 0

print("===== STAGE F: DINO SELF-SIMILARITY =====")
print("Pred cache dir:", pred_dir)
print("Total images:", len(all_rows))

for cid, img_path, split in tqdm(all_rows.itertuples(index=False), total=len(all_rows), desc="DINO self-similarity"):
    outp = _pred_npz_path(cid)
    if Path(outp).exists():
        try:
            z = np.load(outp, allow_pickle=True)
            rows.append({
                "case_id": cid,
                "split": split,
                "confidence": int(z["confidence"]),
                "area_frac": float(z["area_frac"]),
                "npz_path": outp,
            })
            cached += 1
            continue
        except Exception:
            # kalau cache korup, rebuild
            pass

    try:
        feat_dict = get_feat(cid)
        mask_u8, conf, area_frac = dino_selfsim_mask_from_feat(feat_dict, CFG_SIM)
        outp2 = save_pred_dino_raw(cid, mask_u8, conf, area_frac)

        rows.append({
            "case_id": cid,
            "split": split,
            "confidence": int(conf),
            "area_frac": float(area_frac),
            "npz_path": outp2,
        })
        processed += 1
    except Exception as e:
        errors += 1
        if errors <= 5:
            print(f"[ERROR] case_id={cid} split={split} | {type(e).__name__}: {e}")
        # fallback: authentic-like empty mask
        # ambil H0,W0 dari fitur jika bisa
        try:
            feat_dict = get_feat(cid)
            H0, W0 = int(feat_dict["H0"]), int(feat_dict["W0"])
        except Exception:
            H0, W0 = 1, 1
        mask_u8 = np.zeros((H0, W0), np.uint8)
        outp2 = save_pred_dino_raw(cid, mask_u8, 0, 0.0)
        rows.append({
            "case_id": cid,
            "split": split,
            "confidence": 0,
            "area_frac": 0.0,
            "npz_path": outp2,
        })

df_pred_dino_raw = pd.DataFrame(rows)

# merge fold info jika ada
if "fold" in df_index.columns:
    df_pred_dino_raw = df_pred_dino_raw.merge(df_index[["case_id", "fold"]], on="case_id", how="left")

# save artifact
out_csv = os.path.join(PATHS["artifact_dir"], "pred_dino_raw.csv")
df_pred_dino_raw.to_csv(out_csv, index=False)

print("\n===== STAGE F DONE =====")
print(f"Processed (new): {processed} | cached: {cached} | errors: {errors}")
print("Saved:", out_csv)
print(df_pred_dino_raw.head(10))


===== STAGE F: DINO SELF-SIMILARITY =====
Pred cache dir: /kaggle/working/recodai_luc/cache/pred_dino_raw
Total images: 2796


DINO self-similarity:   0%|          | 0/2796 [00:00<?, ?it/s]


===== STAGE F DONE =====
Processed (new): 2796 | cached: 0 | errors: 0
Saved: /kaggle/working/recodai_luc/artifacts/pred_dino_raw.csv
  case_id  split  confidence  area_frac  \
0      10  train          41   0.089111   
1   10015  train          43   0.133436   
2   10017  train          24   0.142529   
3   10030  train          62   0.146471   
4   10070  train          39   0.092700   
5    1008  train         326   0.369089   
6   10138  train          46   0.149871   
7   10139  train          49   0.092071   
8   10147  train          42   0.130762   
9   10152  train         333   0.398369   

                                            npz_path  fold  
0  /kaggle/working/recodai_luc/cache/pred_dino_ra...     1  
1  /kaggle/working/recodai_luc/cache/pred_dino_ra...     0  
2  /kaggle/working/recodai_luc/cache/pred_dino_ra...     1  
3  /kaggle/working/recodai_luc/cache/pred_dino_ra...     2  
4  /kaggle/working/recodai_luc/cache/pred_dino_ra...     4  
5  /kaggle/working/recoda

# Post-processing

In [8]:
# ============================================================
# STAGE G — POST-PROCESSING (DINO RAW -> INSTANCE LIST + UNION) — ONE CELL
# Fokus:
# - Ambil prediksi mask mentah dari Stage F (cache/pred_dino_raw)
# - Rapikan mask (morphology opsional)
# - Pisahkan jadi instance via connected components
# - Filter komponen kecil + batasi jumlah instance
# - Simpan hasil postprocess ke cache/pred_dino_pp
#
# Prasyarat:
# - PATHS, df_index sudah ada
# - Stage F sudah jalan dan membuat cache: cache/pred_dino_raw/{case_id}.npz
#
# Output:
# - cache/pred_dino_pp/{case_id}.npz:
#   * union_mask_u8 (H0,W0) uint8 {0,1}
#   * inst_rles (array of strings)  # RLE per instance
#   * union_rle (string)
#   * n_instances (int)
#   * confidence_raw (int), confidence_pp (int)
#   * area_frac (float)
# - df_pred_dino_pp (train+test): case_id, split, n_instances, confidence_pp, area_frac, npz_path
# - get_pred_dino_pp(case_id) helper
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

# ----------------------------
# Ensure df_index tersedia
# ----------------------------
if "df_index" not in globals() or not isinstance(df_index, pd.DataFrame):
    art = Path(PATHS["artifact_dir"]) / "df_index_with_folds.parquet"
    art_csv = Path(PATHS["artifact_dir"]) / "df_index_with_folds.csv"
    if art.exists():
        df_index = pd.read_parquet(art)
    elif art_csv.exists():
        df_index = pd.read_csv(art_csv)
    else:
        art2 = Path(PATHS["artifact_dir"]) / "df_index.parquet"
        art2_csv = Path(PATHS["artifact_dir"]) / "df_index.csv"
        if art2.exists():
            df_index = pd.read_parquet(art2)
        elif art2_csv.exists():
            df_index = pd.read_csv(art2_csv)
        else:
            raise RuntimeError("df_index tidak ada. Jalankan Stage B/D dulu.")

work_root = PATHS.get("work_root", "/kaggle/working/recodai_luc")
raw_dir = os.path.join(work_root, "cache", "pred_dino_raw")
pp_dir  = os.path.join(work_root, "cache", "pred_dino_pp")
os.makedirs(pp_dir, exist_ok=True)

if not Path(raw_dir).exists():
    raise FileNotFoundError(f"Folder pred_dino_raw tidak ditemukan: {raw_dir}. Jalankan Stage F dulu.")

# ----------------------------
# CONFIG post-process
# ----------------------------
CFG_PP = {
    # morphology (0=off)
    "close_ks": 7,          # isi bolong kecil
    "open_ks": 3,           # buang noise kecil
    "dilate_ks": 0,         # optional, kalau mask terlalu tipis
    "erode_ks": 0,          # optional

    # connected components filter
    "min_area_frac": 0.00035,   # komponen < frac*H*W dibuang
    "max_instances": 6,         # batasi instance (ambil terbesar)
    "min_bbox_side": 6,         # buang komponen yang terlalu kecil (bbox sempit)

    # confidence postprocess (sederhana)
    # confidence_pp = confidence_raw + bonus*(n_instances>=1)
    "conf_bonus": 0,
}

# ----------------------------
# RLE helpers (transpose flatten)
# ----------------------------
def rle_encode(mask_u8):
    pixels = mask_u8.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    if len(runs) == 0:
        return ""
    return " ".join(str(x) for x in runs)

def _morph(mask_u8, close_ks=0, open_ks=0, dilate_ks=0, erode_ks=0):
    m = (mask_u8 > 0).astype(np.uint8)
    if close_ks and close_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_ks, close_ks))
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k, iterations=1)
    if open_ks and open_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_ks, open_ks))
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k, iterations=1)
    if dilate_ks and dilate_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_ks, dilate_ks))
        m = cv2.dilate(m, k, iterations=1)
    if erode_ks and erode_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_ks, erode_ks))
        m = cv2.erode(m, k, iterations=1)
    return m

def _pp_one(mask_u8, confidence_raw, cfg):
    H, W = mask_u8.shape[:2]
    m = _morph(
        mask_u8,
        close_ks=int(cfg["close_ks"]),
        open_ks=int(cfg["open_ks"]),
        dilate_ks=int(cfg["dilate_ks"]),
        erode_ks=int(cfg["erode_ks"]),
    )

    if m.sum() == 0:
        return m, [], 0, float(0.0), int(confidence_raw)

    min_area = int(float(cfg["min_area_frac"]) * H * W)
    num, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)

    comps = []
    for i in range(1, num):
        x = int(stats[i, cv2.CC_STAT_LEFT])
        y = int(stats[i, cv2.CC_STAT_TOP])
        w = int(stats[i, cv2.CC_STAT_WIDTH])
        h = int(stats[i, cv2.CC_STAT_HEIGHT])
        area = int(stats[i, cv2.CC_STAT_AREA])

        if area < min_area:
            continue
        if w < int(cfg["min_bbox_side"]) or h < int(cfg["min_bbox_side"]):
            continue
        comps.append((area, i))

    if not comps:
        # jadi kosong
        z = np.zeros_like(m, dtype=np.uint8)
        return z, [], 0, float(0.0), int(confidence_raw)

    # ambil komponen terbesar
    comps.sort(reverse=True, key=lambda t: t[0])
    comps = comps[: int(cfg["max_instances"])]

    inst_rles = []
    union = np.zeros_like(m, dtype=np.uint8)
    for area, idx in comps:
        inst = (lab == idx).astype(np.uint8)
        if inst.sum() == 0:
            continue
        union = np.maximum(union, inst)
        inst_rles.append(rle_encode(inst))

    # pastikan union konsisten
    union = (union > 0).astype(np.uint8)
    area_frac = float(union.mean())
    conf_pp = int(confidence_raw) + (int(cfg["conf_bonus"]) if len(inst_rles) > 0 else 0)

    return union, inst_rles, len(inst_rles), area_frac, conf_pp

def _raw_npz(case_id):
    return os.path.join(raw_dir, f"{case_id}.npz")

def _pp_npz(case_id):
    return os.path.join(pp_dir, f"{case_id}.npz")

def save_pp(case_id, union_mask_u8, inst_rles, n_instances, area_frac, confidence_raw, confidence_pp):
    outp = _pp_npz(str(case_id))
    union_rle = rle_encode(union_mask_u8) if union_mask_u8.sum() > 0 else ""
    inst_arr = np.array(inst_rles, dtype=object)

    np.savez_compressed(
        outp,
        union_mask_u8=union_mask_u8.astype(np.uint8),
        union_rle=str(union_rle),
        inst_rles=inst_arr,
        n_instances=np.int32(n_instances),
        area_frac=np.float32(area_frac),
        confidence_raw=np.int32(confidence_raw),
        confidence_pp=np.int32(confidence_pp),
    )
    return outp

def get_pred_dino_pp(case_id):
    p = _pp_npz(str(case_id))
    if not Path(p).exists():
        raise FileNotFoundError(f"cache pred_dino_pp tidak ada untuk {case_id}: {p}")
    z = np.load(p, allow_pickle=True)
    return {
        "union_mask_u8": z["union_mask_u8"].astype(np.uint8),
        "union_rle": str(z["union_rle"]),
        "inst_rles": [str(x) for x in z["inst_rles"].tolist()] if "inst_rles" in z.files else [],
        "n_instances": int(z["n_instances"]),
        "area_frac": float(z["area_frac"]),
        "confidence_raw": int(z["confidence_raw"]),
        "confidence_pp": int(z["confidence_pp"]),
        "npz_path": p,
    }

# ----------------------------
# Run post-processing (train+test), skip cached
# ----------------------------
rows = []
all_rows = df_index[["case_id", "split"]].copy()
all_rows["case_id"] = all_rows["case_id"].astype(str)

processed = 0
cached = 0
errors = 0

print("===== STAGE G: POST-PROCESSING =====")
print("raw_dir:", raw_dir)
print("pp_dir :", pp_dir)
print("Total images:", len(all_rows))

for cid, split in tqdm(all_rows.itertuples(index=False), total=len(all_rows), desc="Post-process"):
    outp = _pp_npz(cid)
    if Path(outp).exists():
        try:
            z = np.load(outp, allow_pickle=True)
            rows.append({
                "case_id": cid,
                "split": split,
                "n_instances": int(z["n_instances"]),
                "confidence_pp": int(z["confidence_pp"]),
                "area_frac": float(z["area_frac"]),
                "npz_path": outp,
            })
            cached += 1
            continue
        except Exception:
            pass

    rawp = _raw_npz(cid)
    if not Path(rawp).exists():
        # fallback: kosong
        union = np.zeros((1, 1), np.uint8)
        outp2 = save_pp(cid, union, [], 0, 0.0, 0, 0)
        rows.append({
            "case_id": cid,
            "split": split,
            "n_instances": 0,
            "confidence_pp": 0,
            "area_frac": 0.0,
            "npz_path": outp2,
        })
        processed += 1
        continue

    try:
        z = np.load(rawp, allow_pickle=True)
        mask_u8 = z["mask_u8"].astype(np.uint8)
        conf_raw = int(z["confidence"]) if "confidence" in z.files else 0

        union, inst_rles, n_inst, area_frac, conf_pp = _pp_one(mask_u8, conf_raw, CFG_PP)
        outp2 = save_pp(cid, union, inst_rles, n_inst, area_frac, conf_raw, conf_pp)

        rows.append({
            "case_id": cid,
            "split": split,
            "n_instances": int(n_inst),
            "confidence_pp": int(conf_pp),
            "area_frac": float(area_frac),
            "npz_path": outp2,
        })
        processed += 1
    except Exception as e:
        errors += 1
        if errors <= 5:
            print(f"[ERROR] case_id={cid} split={split} | {type(e).__name__}: {e}")
        # fallback kosong
        union = np.zeros((1, 1), np.uint8)
        outp2 = save_pp(cid, union, [], 0, 0.0, 0, 0)
        rows.append({
            "case_id": cid,
            "split": split,
            "n_instances": 0,
            "confidence_pp": 0,
            "area_frac": 0.0,
            "npz_path": outp2,
        })

df_pred_dino_pp = pd.DataFrame(rows)

# merge fold info jika ada
if "fold" in df_index.columns:
    df_pred_dino_pp = df_pred_dino_pp.merge(df_index[["case_id", "fold"]], on="case_id", how="left")

# save artifact
out_csv = os.path.join(PATHS["artifact_dir"], "pred_dino_pp.csv")
df_pred_dino_pp.to_csv(out_csv, index=False)

print("\n===== STAGE G DONE =====")
print(f"Processed (new): {processed} | cached: {cached} | errors: {errors}")
print("Saved:", out_csv)
print(df_pred_dino_pp.head(10))


===== STAGE G: POST-PROCESSING =====
raw_dir: /kaggle/working/recodai_luc/cache/pred_dino_raw
pp_dir : /kaggle/working/recodai_luc/cache/pred_dino_pp
Total images: 2796


Post-process:   0%|          | 0/2796 [00:00<?, ?it/s]


===== STAGE G DONE =====
Processed (new): 2796 | cached: 0 | errors: 0
Saved: /kaggle/working/recodai_luc/artifacts/pred_dino_pp.csv
  case_id  split  n_instances  confidence_pp  area_frac  \
0      10  train            6             41   0.016195   
1   10015  train            6             43   0.038427   
2   10017  train            6             24   0.092627   
3   10030  train            6             62   0.061111   
4   10070  train            6             39   0.026350   
5    1008  train            6            326   0.302203   
6   10138  train            6             46   0.068465   
7   10139  train            6             49   0.020395   
8   10147  train            6             42   0.049590   
9   10152  train            6            333   0.327638   

                                            npz_path  fold  
0  /kaggle/working/recodai_luc/cache/pred_dino_pp...     1  
1  /kaggle/working/recodai_luc/cache/pred_dino_pp...     0  
2  /kaggle/working/recodai_luc/ca

# Threshold tuning (authentic vs forged + mask threshold)

In [9]:
# ============================================================
# STAGE H — THRESHOLD TUNING (authentic vs forged + mask threshold) — ONE CELL
# Yang ditune (tanpa re-run DINO):
# - thr_conf      : minimal confidence_pp agar dianggap forged
# - thr_area_frac : minimal area_frac mask agar dianggap forged
# - (opsional) n_instances>=1 sudah dipakai sebagai guard
#
# Metric tuning (CV by fold) pakai UNION mask:
# - Pixel-Dice (F1 pixel) dan IoU
# - Image-level F1 (forged detection) sebagai diagnostik
#
# Prasyarat:
# - df_index sudah ada (Stage B/D)
# - load_gt(case_id) sudah ada (Stage C)
# - get_pred_dino_pp(case_id) sudah ada (Stage G)
#
# Output:
# - artifacts/tune_thresholds.csv  (ranking hasil tuning)
# - artifacts/cfg_tune.json        (threshold terbaik)
# - CFG_TUNE (dict) di memori
# ============================================================

import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

# ----------------------------
# Ensure df_index
# ----------------------------
if "df_index" not in globals() or not isinstance(df_index, pd.DataFrame):
    art = Path(PATHS["artifact_dir"]) / "df_index_with_folds.parquet"
    art_csv = Path(PATHS["artifact_dir"]) / "df_index_with_folds.csv"
    if art.exists():
        df_index = pd.read_parquet(art)
    elif art_csv.exists():
        df_index = pd.read_csv(art_csv)
    else:
        raise RuntimeError("df_index tidak ada. Jalankan Stage D dulu.")

df_train = df_index[df_index["split"].eq("train")].reset_index(drop=True)

# pastikan fold ada
if "fold" not in df_train.columns:
    raise RuntimeError("Kolom fold tidak ada. Jalankan Stage D dulu (validasi internal).")

# pastikan helper ada
if "load_gt" not in globals() or not callable(load_gt):
    raise RuntimeError("load_gt(case_id) tidak ada. Jalankan Stage C dulu.")
if "get_pred_dino_pp" not in globals() or not callable(get_pred_dino_pp):
    raise RuntimeError("get_pred_dino_pp(case_id) tidak ada. Jalankan Stage G dulu.")

# ----------------------------
# Precompute confusion per image (sekali saja) agar tuning cepat
# ----------------------------
def _align_nearest(mask_u8, H, W):
    if mask_u8.shape[0] == H and mask_u8.shape[1] == W:
        return mask_u8
    return cv2.resize(mask_u8.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(np.uint8)

case_ids = df_train["case_id"].astype(str).tolist()
folds = df_train["fold"].astype(int).to_numpy()

tp_list = []
fp_list = []
gt_area_list = []
pred_area_list = []
conf_list = []
area_frac_list = []
ninst_list = []

bad = 0
print("Precompute TP/FP/GT_area per train image (union mask)...")

for cid in tqdm(case_ids, desc="Precompute"):
    try:
        gt_union, _ = load_gt(cid)
        gt_union = (gt_union > 0).astype(np.uint8)
        H, W = gt_union.shape[:2]
        gt_area = int(gt_union.sum())

        pred = get_pred_dino_pp(cid)
        pm = pred["union_mask_u8"]
        pm = (pm > 0).astype(np.uint8)
        pm = _align_nearest(pm, H, W)

        pred_area = int(pm.sum())
        tp = int((pm & gt_union).sum())
        fp = int(pred_area - tp)

        tp_list.append(tp)
        fp_list.append(fp)
        gt_area_list.append(gt_area)
        pred_area_list.append(pred_area)
        conf_list.append(int(pred["confidence_pp"]))
        area_frac_list.append(float(pred["area_frac"]))
        ninst_list.append(int(pred["n_instances"]))
    except Exception as e:
        bad += 1
        # fallback: anggap kosong
        tp_list.append(0)
        fp_list.append(0)
        gt_area_list.append(0)
        pred_area_list.append(0)
        conf_list.append(0)
        area_frac_list.append(0.0)
        ninst_list.append(0)
        if bad <= 3:
            print(f"[WARN] case_id={cid} | {type(e).__name__}: {e}")

tp_arr = np.asarray(tp_list, dtype=np.float64)
fp_arr = np.asarray(fp_list, dtype=np.float64)
gt_area_arr = np.asarray(gt_area_list, dtype=np.float64)
pred_area_arr = np.asarray(pred_area_list, dtype=np.float64)
conf_arr = np.asarray(conf_list, dtype=np.float64)
area_frac_arr = np.asarray(area_frac_list, dtype=np.float64)
ninst_arr = np.asarray(ninst_list, dtype=np.int32)

print(f"Done. bad={bad}/{len(case_ids)}")

# ----------------------------
# Build threshold grids (berdasarkan distribusi data agar tidak ngarang)
# ----------------------------
def _unique_sorted_int(vals):
    vals = [int(x) for x in vals]
    vals = sorted(set(vals))
    return vals

def _unique_sorted_float(vals):
    vals = [float(x) for x in vals]
    vals = sorted(set(vals))
    return vals

# grid confidence: gabung beberapa quantile + nilai aman
conf_nonzero = conf_arr[conf_arr > 0]
if len(conf_nonzero) == 0:
    conf_grid = [0]
else:
    qs = np.quantile(conf_nonzero, [0.05, 0.15, 0.30, 0.50, 0.70, 0.85, 0.95])
    base = [0, 2, 4, 6, 8, 10, 12, 16, 20, 24, 32, 40]
    conf_grid = _unique_sorted_int(list(qs) + base)

# grid area_frac: fokus pada pred non-empty
af_nonzero = area_frac_arr[area_frac_arr > 0]
if len(af_nonzero) == 0:
    area_grid = [0.0]
else:
    qs = np.quantile(af_nonzero, [0.05, 0.15, 0.30, 0.50, 0.70, 0.85, 0.95])
    base = [0.0, 0.0001, 0.0002, 0.00035, 0.0005, 0.0008, 0.0010, 0.0015, 0.0020]
    # bulatkan supaya grid tidak terlalu banyak
    area_grid = _unique_sorted_float(list(np.round(qs, 6)) + base)

# Guard agar grid tidak terlalu besar (CPU)
conf_grid = conf_grid[:25]
area_grid = area_grid[:25]

print("\n===== GRID =====")
print("conf_grid:", conf_grid)
print("area_grid:", area_grid)

# ----------------------------
# Evaluator per fold
# ----------------------------
def safe_div(a, b):
    return float(a / b) if b > 0 else 0.0

def eval_one_setting(thr_conf, thr_area_frac):
    # keep = dianggap forged (mask dipakai)
    keep = (conf_arr >= thr_conf) & (area_frac_arr >= thr_area_frac) & (ninst_arr >= 1) & (pred_area_arr > 0)

    folds_unique = np.unique(folds)
    dice_folds = []
    iou_folds = []
    imgf1_folds = []

    for f in folds_unique:
        idx = (folds == f)
        if idx.sum() == 0:
            continue

        k = keep[idx]

        TP = float((tp_arr[idx] * k).sum())
        FP = float((fp_arr[idx] * k).sum())
        GT = float(gt_area_arr[idx].sum())
        # FN = GT - TP (karena kalau tidak keep, TP=0)
        FN = GT - TP

        dice = safe_div(2.0 * TP, (2.0 * TP + FP + FN))
        iou  = safe_div(TP, (TP + FP + FN))

        # image-level F1 (pred positive jika keep)
        y_true = (gt_area_arr[idx] > 0)
        y_pred = k

        tp_img = float(np.logical_and(y_true, y_pred).sum())
        fp_img = float(np.logical_and(~y_true, y_pred).sum())
        fn_img = float(np.logical_and(y_true, ~y_pred).sum())
        img_f1 = safe_div(2.0 * tp_img, (2.0 * tp_img + fp_img + fn_img))

        dice_folds.append(dice)
        iou_folds.append(iou)
        imgf1_folds.append(img_f1)

    return {
        "dice_mean": float(np.mean(dice_folds)) if dice_folds else 0.0,
        "iou_mean": float(np.mean(iou_folds)) if iou_folds else 0.0,
        "img_f1_mean": float(np.mean(imgf1_folds)) if imgf1_folds else 0.0,
    }

# ----------------------------
# Grid search
# ----------------------------
results = []
best = None

print("\nRunning grid search...")
for thr_conf in tqdm(conf_grid, desc="thr_conf"):
    for thr_area in area_grid:
        m = eval_one_setting(thr_conf, thr_area)
        row = {
            "thr_conf": int(thr_conf),
            "thr_area_frac": float(thr_area),
            **m
        }
        results.append(row)
        if best is None or row["dice_mean"] > best["dice_mean"]:
            best = row

df_tune = pd.DataFrame(results).sort_values(["dice_mean", "img_f1_mean", "iou_mean"], ascending=False).reset_index(drop=True)

# ----------------------------
# Save artifacts
# ----------------------------
out_csv = os.path.join(PATHS["artifact_dir"], "tune_thresholds.csv")
df_tune.to_csv(out_csv, index=False)

CFG_TUNE = {
    "thr_conf": int(best["thr_conf"]),
    "thr_area_frac": float(best["thr_area_frac"]),
    "metric": {
        "dice_mean": float(best["dice_mean"]),
        "iou_mean": float(best["iou_mean"]),
        "img_f1_mean": float(best["img_f1_mean"]),
    }
}

out_json = os.path.join(PATHS["artifact_dir"], "cfg_tune.json")
with open(out_json, "w") as f:
    json.dump(CFG_TUNE, f, indent=2)

print("\n===== BEST THRESHOLDS =====")
print(CFG_TUNE)
print("\nTop 10:")
print(df_tune.head(10))
print("\nSaved:")
print(out_csv)
print(out_json)


Precompute TP/FP/GT_area per train image (union mask)...


Precompute:   0%|          | 0/2795 [00:00<?, ?it/s]

Done. bad=0/2795

===== GRID =====
conf_grid: [0, 2, 4, 6, 8, 10, 12, 16, 18, 20, 24, 32, 40, 41, 49, 58, 75, 223]
area_grid: [0.0, 0.0001, 0.0002, 0.00035, 0.0005, 0.0008, 0.001, 0.0015, 0.002, 0.029413, 0.037981, 0.047747, 0.0639, 0.091687, 0.1339, 0.297425]

Running grid search...


thr_conf:   0%|          | 0/18 [00:00<?, ?it/s]


===== BEST THRESHOLDS =====
{'thr_conf': 0, 'thr_area_frac': 0.0, 'metric': {'dice_mean': 0.08346958704665734, 'iou_mean': 0.04356393102564721, 'img_f1_mean': 0.9978563239757795}}

Top 10:
   thr_conf  thr_area_frac  dice_mean  iou_mean  img_f1_mean
0         0        0.00000    0.08347  0.043564     0.997856
1         0        0.00010    0.08347  0.043564     0.997856
2         0        0.00020    0.08347  0.043564     0.997856
3         0        0.00035    0.08347  0.043564     0.997856
4         0        0.00050    0.08347  0.043564     0.997856
5         0        0.00080    0.08347  0.043564     0.997856
6         0        0.00100    0.08347  0.043564     0.997856
7         0        0.00150    0.08347  0.043564     0.997856
8         0        0.00200    0.08347  0.043564     0.997856
9         2        0.00000    0.08347  0.043564     0.997856

Saved:
/kaggle/working/recodai_luc/artifacts/tune_thresholds.csv
/kaggle/working/recodai_luc/artifacts/cfg_tune.json


# Ensemble

In [10]:
# ============================================================
# STAGE I — ENSEMBLE (DINO-PP + CLASSIC) — ONE CELL
# Tujuan:
# - Menggabungkan prediksi DINO (Stage G + threshold Stage H) dengan baseline klasik (Stage E1)
# - Output berupa mask final + instance RLE list (untuk dipakai di tahap submission)
#
# Prasyarat:
# - df_index sudah ada (Stage B/D)
# - get_pred_dino_pp(case_id) ada (Stage G)
# - artifacts/cfg_tune.json ada (Stage H)
# - artifacts/pred_classic.csv ada (Stage E1)
#
# Output:
# - cache/pred_ens/{case_id}.npz:
#   * union_mask_u8, union_rle, inst_rles, n_instances, source_used
# - artifacts/pred_ensemble.csv
# - df_pred_ensemble (DataFrame) di memori
# ============================================================

import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

# ----------------------------
# Ensure df_index
# ----------------------------
if "df_index" not in globals() or not isinstance(df_index, pd.DataFrame):
    art = Path(PATHS["artifact_dir"]) / "df_index_with_folds.parquet"
    art_csv = Path(PATHS["artifact_dir"]) / "df_index_with_folds.csv"
    if art.exists():
        df_index = pd.read_parquet(art)
    elif art_csv.exists():
        df_index = pd.read_csv(art_csv)
    else:
        art2 = Path(PATHS["artifact_dir"]) / "df_index.parquet"
        art2_csv = Path(PATHS["artifact_dir"]) / "df_index.csv"
        if art2.exists():
            df_index = pd.read_parquet(art2)
        elif art2_csv.exists():
            df_index = pd.read_csv(art2_csv)
        else:
            raise RuntimeError("df_index tidak ada. Jalankan Stage B dulu.")

# pastikan helper DINO postprocess ada
if "get_pred_dino_pp" not in globals() or not callable(get_pred_dino_pp):
    raise RuntimeError("get_pred_dino_pp(case_id) tidak ada. Jalankan Stage G dulu.")

work_root = PATHS.get("work_root", "/kaggle/working/recodai_luc")
ens_dir = os.path.join(work_root, "cache", "pred_ens")
os.makedirs(ens_dir, exist_ok=True)

# ----------------------------
# Load thresholds (Stage H)
# ----------------------------
cfg_tune_path = Path(PATHS["artifact_dir"]) / "cfg_tune.json"
if not cfg_tune_path.exists():
    raise FileNotFoundError(f"cfg_tune.json tidak ditemukan: {cfg_tune_path} (jalankan Stage H dulu)")
with open(cfg_tune_path, "r") as f:
    CFG_TUNE = json.load(f)

THR_CONF = int(CFG_TUNE.get("thr_conf", 0))
THR_AREA = float(CFG_TUNE.get("thr_area_frac", 0.0))

print("===== ENSEMBLE CONFIG =====")
print("thr_conf:", THR_CONF, "| thr_area_frac:", THR_AREA)

# ----------------------------
# Load classic predictions (Stage E1)
# ----------------------------
classic_path = Path(PATHS["artifact_dir"]) / "pred_classic.csv"
if not classic_path.exists():
    raise FileNotFoundError(f"pred_classic.csv tidak ditemukan: {classic_path} (jalankan Stage E1 dulu)")

df_classic = pd.read_csv(classic_path)
df_classic["case_id"] = df_classic["case_id"].astype(str)
classic_anno = dict(zip(df_classic["case_id"], df_classic["annotation_pred"].astype(str)))
classic_conf = dict(zip(df_classic["case_id"], df_classic.get("confidence", pd.Series([0]*len(df_classic))).fillna(0).astype(int)))

# ----------------------------
# RLE encode/decode (transpose flatten)
# ----------------------------
def rle_encode(mask_u8):
    pixels = mask_u8.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    if len(runs) == 0:
        return ""
    return " ".join(str(x) for x in runs)

def rle_decode(rle, H, W):
    if rle is None:
        return np.zeros((H, W), np.uint8)
    s = str(rle).strip()
    if s == "" or s.lower() == "authentic" or s.lower() == "nan":
        return np.zeros((H, W), np.uint8)
    nums = s.split()
    if len(nums) < 2 or (len(nums) % 2 != 0):
        return np.zeros((H, W), np.uint8)
    nums = np.array([int(x) for x in nums], dtype=np.int64)
    starts = nums[0::2] - 1
    lens = nums[1::2]
    flat = np.zeros(H * W, dtype=np.uint8)
    for st, ln in zip(starts, lens):
        if ln <= 0:
            continue
        st = int(max(0, st))
        ed = int(min(H * W, st + ln))
        flat[st:ed] = 1
    # because encoding used mask.T.flatten()
    return flat.reshape((W, H)).T

# ----------------------------
# Postprocess instances (CC + filter)
# ----------------------------
CFG_ENS_PP = {
    "close_ks": 5,
    "open_ks": 3,
    "min_area_frac": 0.00035,
    "max_instances": 6,
    "min_bbox_side": 6,
}

def morph(mask_u8, close_ks=0, open_ks=0):
    m = (mask_u8 > 0).astype(np.uint8)
    if close_ks and close_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_ks, close_ks))
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k, iterations=1)
    if open_ks and open_ks > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_ks, open_ks))
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k, iterations=1)
    return m

def union_to_instances(union_mask_u8, cfg):
    H, W = union_mask_u8.shape[:2]
    m = morph(union_mask_u8, close_ks=int(cfg["close_ks"]), open_ks=int(cfg["open_ks"]))
    if m.sum() == 0:
        return m, []

    min_area = int(float(cfg["min_area_frac"]) * H * W)
    num, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)

    comps = []
    for i in range(1, num):
        x = int(stats[i, cv2.CC_STAT_LEFT])
        y = int(stats[i, cv2.CC_STAT_TOP])
        w = int(stats[i, cv2.CC_STAT_WIDTH])
        h = int(stats[i, cv2.CC_STAT_HEIGHT])
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        if w < int(cfg["min_bbox_side"]) or h < int(cfg["min_bbox_side"]):
            continue
        comps.append((area, i))

    if not comps:
        z = np.zeros_like(m, dtype=np.uint8)
        return z, []

    comps.sort(reverse=True, key=lambda t: t[0])
    comps = comps[: int(cfg["max_instances"])]

    inst_rles = []
    union = np.zeros_like(m, dtype=np.uint8)
    for area, idx in comps:
        inst = (lab == idx).astype(np.uint8)
        if inst.sum() == 0:
            continue
        union = np.maximum(union, inst)
        inst_rles.append(rle_encode(inst))

    union = (union > 0).astype(np.uint8)
    return union, inst_rles

def ens_npz_path(case_id):
    return os.path.join(ens_dir, f"{case_id}.npz")

def save_ens(case_id, union_mask_u8, inst_rles, source_used, conf_dino, conf_classic):
    p = ens_npz_path(str(case_id))
    np.savez_compressed(
        p,
        union_mask_u8=union_mask_u8.astype(np.uint8),
        union_rle=str(rle_encode(union_mask_u8) if union_mask_u8.sum() > 0 else ""),
        inst_rles=np.array(inst_rles, dtype=object),
        n_instances=np.int32(len(inst_rles)),
        source_used=str(source_used),
        conf_dino=np.int32(conf_dino),
        conf_classic=np.int32(conf_classic),
        area_frac=np.float32(float(union_mask_u8.mean()) if union_mask_u8.size else 0.0),
    )
    return p

# ----------------------------
# ENSEMBLE LOOP (train+test)
# ----------------------------
rows = []
all_rows = df_index[["case_id", "split"]].copy()
all_rows["case_id"] = all_rows["case_id"].astype(str)

print("\n===== RUN ENSEMBLE =====")
for cid, split in tqdm(all_rows.itertuples(index=False), total=len(all_rows), desc="Ensemble"):
    outp = ens_npz_path(cid)
    if Path(outp).exists():
        # quick load summary
        z = np.load(outp, allow_pickle=True)
        rows.append({
            "case_id": cid,
            "split": split,
            "source_used": str(z["source_used"]),
            "n_instances": int(z["n_instances"]),
            "area_frac": float(z["area_frac"]),
            "conf_dino": int(z["conf_dino"]),
            "conf_classic": int(z["conf_classic"]),
            "npz_path": outp,
        })
        continue

    # 1) DINO gating
    d = get_pred_dino_pp(cid)
    H, W = d["union_mask_u8"].shape[:2]
    dino_keep = (int(d["confidence_pp"]) >= THR_CONF) and (float(d["area_frac"]) >= THR_AREA) and (int(d["n_instances"]) >= 1)
    dino_union = d["union_mask_u8"].astype(np.uint8) if dino_keep else np.zeros((H, W), np.uint8)

    # 2) Classic mask (decode RLE) + gating sederhana
    c_anno = classic_anno.get(cid, "authentic")
    c_conf = int(classic_conf.get(cid, 0))
    classic_union = rle_decode(c_anno, H, W)  # already union
    classic_keep = classic_union.sum() > 0

    # 3) Combine
    if dino_keep and classic_keep:
        union = np.maximum(dino_union, classic_union)
        source = "both"
    elif dino_keep:
        union = dino_union
        source = "dino"
    elif classic_keep:
        union = classic_union
        source = "classic"
    else:
        union = np.zeros((H, W), np.uint8)
        source = "none"

    # 4) Instance rebuild (biar konsisten dan bisa memecah union yang nempel)
    union_pp, inst_rles = union_to_instances(union, CFG_ENS_PP)

    outp2 = save_ens(
        cid,
        union_pp,
        inst_rles,
        source_used=source,
        conf_dino=int(d["confidence_pp"]),
        conf_classic=int(c_conf),
    )

    rows.append({
        "case_id": cid,
        "split": split,
        "source_used": source,
        "n_instances": len(inst_rles),
        "area_frac": float(union_pp.mean()) if union_pp.size else 0.0,
        "conf_dino": int(d["confidence_pp"]),
        "conf_classic": int(c_conf),
        "npz_path": outp2,
    })

df_pred_ensemble = pd.DataFrame(rows)

# merge fold if available
if "fold" in df_index.columns:
    df_pred_ensemble = df_pred_ensemble.merge(df_index[["case_id", "fold"]], on="case_id", how="left")

# save artifact
out_csv = os.path.join(PATHS["artifact_dir"], "pred_ensemble.csv")
df_pred_ensemble.to_csv(out_csv, index=False)

print("\n===== ENSEMBLE DONE =====")
print("Saved:", out_csv)
print(df_pred_ensemble.head(10))


===== ENSEMBLE CONFIG =====
thr_conf: 0 | thr_area_frac: 0.0

===== RUN ENSEMBLE =====


Ensemble:   0%|          | 0/2796 [00:00<?, ?it/s]


===== ENSEMBLE DONE =====
Saved: /kaggle/working/recodai_luc/artifacts/pred_ensemble.csv
  case_id  split source_used  n_instances  area_frac  conf_dino  conf_classic  \
0      10  train        dino            6   0.016195         41             0   
1   10015  train        dino            6   0.038427         43             0   
2   10017  train        dino            6   0.092627         24             0   
3   10030  train        both            6   0.168104         62            31   
4   10070  train        dino            6   0.026350         39             0   
5    1008  train        dino            6   0.302203        326             0   
6   10138  train        dino            6   0.068465         46             0   
7   10139  train        dino            6   0.020395         49             0   
8   10147  train        dino            6   0.049590         42             0   
9   10152  train        dino            6   0.327638        333             0   

                  

# RLE encoding & format submission (submission.csv)

In [11]:
# ============================================================
# STAGE J — RLE ENCODING & FORMAT SUBMISSION (submission.csv) — REVISI FULL
# Penting (Kaggle):
# - File HARUS bernama tepat: submission.csv
# - Dan HARUS berada langsung di: /kaggle/working/submission.csv
#
# Prasyarat:
# - PATHS sudah ada
# - cache ensemble ada: /kaggle/working/recodai_luc/cache/pred_ens/{case_id}.npz
# - sample_submission.csv ada
#
# Output:
# - /kaggle/working/submission.csv   (ini yang dibaca Kaggle)
# - (opsional) /kaggle/working/recodai_luc/outputs/submission.csv (copy)
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Ensure PATHS + df_index tersedia
# ----------------------------
if "PATHS" not in globals() or not isinstance(PATHS, dict):
    raise RuntimeError("PATHS tidak ada. Jalankan Stage B dulu.")

if "df_index" not in globals() or not isinstance(df_index, pd.DataFrame):
    art = Path(PATHS["artifact_dir"]) / "df_index_with_folds.parquet"
    art_csv = Path(PATHS["artifact_dir"]) / "df_index_with_folds.csv"
    if art.exists():
        df_index = pd.read_parquet(art)
    elif art_csv.exists():
        df_index = pd.read_csv(art_csv)
    else:
        art2 = Path(PATHS["artifact_dir"]) / "df_index.parquet"
        art2_csv = Path(PATHS["artifact_dir"]) / "df_index.csv"
        if art2.exists():
            df_index = pd.read_parquet(art2)
        elif art2_csv.exists():
            df_index = pd.read_csv(art2_csv)
        else:
            raise RuntimeError("df_index tidak ada. Jalankan Stage B/D dulu.")

# ----------------------------
# Paths
# ----------------------------
work_root = PATHS.get("work_root", "/kaggle/working/recodai_luc")
ens_dir = os.path.join(work_root, "cache", "pred_ens")   # stage I output
copy_dir = os.path.join(work_root, "outputs")
os.makedirs(copy_dir, exist_ok=True)

# Kaggle REQUIRED output location:
OUT_MAIN = "/kaggle/working/submission.csv"
OUT_COPY = os.path.join(copy_dir, "submission.csv")

sample_path = PATHS.get("sample_submission", f"{PATHS['root']}/sample_submission.csv")
if not Path(sample_path).exists():
    raise FileNotFoundError(f"sample_submission.csv tidak ditemukan: {sample_path}")

# ----------------------------
# Load sample submission (final ordering)
# ----------------------------
sub = pd.read_csv(sample_path)
need_cols = {"case_id", "annotation"}
if not need_cols.issubset(set(sub.columns)):
    raise ValueError(f"sample_submission harus punya kolom {need_cols}. Kolom ada: {list(sub.columns)}")

sub["case_id"] = sub["case_id"].astype(str)

# ----------------------------
# Helper load ensemble npz
# ----------------------------
def ens_npz(case_id: str) -> str:
    return os.path.join(ens_dir, f"{case_id}.npz")

def safe_str(x):
    # handle numpy scalar / bytes
    try:
        if isinstance(x, bytes):
            return x.decode("utf-8", errors="ignore")
    except Exception:
        pass
    return str(x)

# ----------------------------
# Build predictions
# ----------------------------
pred_map = {}
missing = 0
bad = 0

ens_exists = Path(ens_dir).exists()
if not ens_exists:
    print(f"[WARN] Folder ensemble tidak ditemukan: {ens_dir}")
    print("Semua test akan diisi 'authentic' supaya submission.csv tetap terbentuk.")

for cid in sub["case_id"].tolist():
    if not ens_exists:
        pred_map[cid] = "authentic"
        continue

    p = ens_npz(cid)
    if not Path(p).exists():
        pred_map[cid] = "authentic"
        missing += 1
        continue

    try:
        z = np.load(p, allow_pickle=True)
        union_rle = ""
        if "union_rle" in z.files:
            union_rle = safe_str(z["union_rle"])
        union_rle = union_rle.strip()

        if union_rle == "" or union_rle.lower() in ("nan", "none"):
            pred_map[cid] = "authentic"
        else:
            pred_map[cid] = union_rle
    except Exception:
        bad += 1
        pred_map[cid] = "authentic"

sub["annotation"] = sub["case_id"].map(lambda x: pred_map.get(str(x), "authentic"))

# ----------------------------
# Write Kaggle-required output
# ----------------------------
sub.to_csv(OUT_MAIN, index=False)

# optional copy
try:
    sub.to_csv(OUT_COPY, index=False)
except Exception:
    pass

# ----------------------------
# Verify output exists (ini yang dicek Kaggle)
# ----------------------------
print("===== SUBMISSION READY =====")
print("Wrote main :", OUT_MAIN, "| exists:", Path(OUT_MAIN).exists(), "| size:", Path(OUT_MAIN).stat().st_size if Path(OUT_MAIN).exists() else 0)
print("Wrote copy :", OUT_COPY, "| exists:", Path(OUT_COPY).exists())
print("Rows:", len(sub), "| missing ensemble cache:", missing, "| bad npz:", bad)
print(sub.head(10))


===== SUBMISSION READY =====
Wrote main : /kaggle/working/submission.csv | exists: True | size: 31404
Wrote copy : /kaggle/working/recodai_luc/outputs/submission.csv | exists: True
Rows: 1 | missing ensemble cache: 0 | bad npz: 0
  case_id                                         annotation
0      45  170591 82 170792 82 172034 84 172235 84 173477...
